# 冻结方法：UHDM与DCID小规模test验证
不训练。需要GPU、Internet、UHDM数据集、moire_frozen_test_models.zip。DCID按锁定版本仅获取20对test图像。

In [ ]:
%pip install -q gdown pyyaml scikit-image huggingface_hub


In [ ]:
import types
I=types.ModuleType("independent_helpers")
exec('import types\nS = types.ModuleType(\'spatial_helpers\')\nexec(\'import types\\nC = types.ModuleType(\\\'calibration_helpers\\\')\\nexec(\\\'import types\\\\nJ = types.ModuleType(\\\\\\\'joint_helpers\\\\\\\')\\\\nexec(\\\\\\\'import types\\\\\\\\nH = types.ModuleType(\\\\\\\\\\\\\\\'baseline_helpers\\\\\\\\\\\\\\\')\\\\\\\\nexec(\\\\\\\\\\\\\\\'"""复制整个文件到 Kaggle 新单元格执行。先安装 gdown pyyaml scikit-image。\\\\\\\\\\\\\\\\n独立入口：不依赖原 UHDM 数据集、不加载融合模型、不训练。\\\\\\\\\\\\\\\\n"""\\\\\\\\\\\\\\\\nfrom pathlib import Path\\\\\\\\\\\\\\\\nimport csv, io, json, hashlib, zipfile, sys, subprocess, importlib.util, time\\\\\\\\\\\\\\\\nfrom datetime import datetime\\\\\\\\\\\\\\\\nimport numpy as np\\\\\\\\\\\\\\\\nimport pandas as pd\\\\\\\\\\\\\\\\nfrom PIL import Image, ImageOps, ImageDraw\\\\\\\\\\\\\\\\nimport torch\\\\\\\\\\\\\\\\nimport torch.nn.functional as F\\\\\\\\\\\\\\\\nimport yaml\\\\\\\\\\\\\\\\nfrom skimage.metrics import structural_similarity\\\\\\\\\\\\\\\\nfrom tqdm.auto import tqdm\\\\\\\\\\\\\\\\nfrom IPython.display import display, FileLink\\\\\\\\\\\\\\\\nDATA_SOURCE = None\\\\\\\\\\\\\\\\nLOCAL_WEIGHT = None\\\\\\\\\\\\\\\\nREGION = 768\\\\\\\\\\\\\\\\nSCALES = (1.0, 0.5)\\\\\\\\\\\\\\\\nMAX_PAIRS = 20\\\\\\\\\\\\\\\\nROOT = Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/working/dcid_baseline\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\ndef sha_file(path):\\\\\\\\\\\\\\\\n    h = hashlib.sha256()\\\\\\\\\\\\\\\\n    with Path(path).open(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'rb\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\') as f:\\\\\\\\\\\\\\\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'):\\\\\\\\\\\\\\\\n            h.update(block)\\\\\\\\\\\\\\\\n    return h.hexdigest()\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\nclass PairSource:\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\n    def __init__(self, source):\\\\\\\\\\\\\\\\n        self.path = Path(source)\\\\\\\\\\\\\\\\n        self.archive = zipfile.ZipFile(self.path) if self.path.suffix.lower() == \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'.zip\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\' else None\\\\\\\\\\\\\\\\n        if self.archive:\\\\\\\\\\\\\\\\n            members = [n for n in self.archive.namelist() if n.endswith(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'_manifest_draft.csv\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')]\\\\\\\\\\\\\\\\n            if len(members) != 1:\\\\\\\\\\\\\\\\n                raise RuntimeError(f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'ZIP 内应有唯一原始清单，实际为 {members}\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n            self.manifest_bytes = self.archive.read(members[0])\\\\\\\\\\\\\\\\n            self.prefix = members[0].rsplit(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', 1)[0] + \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\' if \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\' in members[0] else \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\n        else:\\\\\\\\\\\\\\\\n            self.manifest_bytes = self.path.read_bytes()\\\\\\\\\\\\\\\\n        self.rows = list(csv.DictReader(io.StringIO(self.manifest_bytes.decode(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'utf-8-sig\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), newline=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')))\\\\\\\\\\\\\\\\n        if not self.rows:\\\\\\\\\\\\\\\\n            raise RuntimeError(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'清单为空\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        if not {\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'target\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'} <= set(self.rows[0]):\\\\\\\\\\\\\\\\n            raise RuntimeError(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'清单缺少必要字段\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        ids = [r[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'] for r in self.rows]\\\\\\\\\\\\\\\\n        if len(ids) != len(set(ids)):\\\\\\\\\\\\\\\\n            raise RuntimeError(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id 重复，请检查清单\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\n    def read_bytes(self, row, key):\\\\\\\\\\\\\\\\n        name = row[key].replace(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        relative = \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'images/\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\' + name.split(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/images/\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', 1)[1] if \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/images/\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\' in name else name\\\\\\\\\\\\\\\\n        if self.archive:\\\\\\\\\\\\\\\\n            return self.archive.read(self.prefix + relative)\\\\\\\\\\\\\\\\n        p = Path(name)\\\\\\\\\\\\\\\\n        if not p.is_file():\\\\\\\\\\\\\\\\n            p = self.path.parent / relative\\\\\\\\\\\\\\\\n        if not p.is_file():\\\\\\\\\\\\\\\\n            raise FileNotFoundError(f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'{key}: {name}，也未找到 {p}\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        return p.read_bytes()\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\n    def read(self, row, key):\\\\\\\\\\\\\\\\n        with Image.open(io.BytesIO(self.read_bytes(row, key))) as image:\\\\\\\\\\\\\\\\n            return np.array(image.convert(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'RGB\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), copy=True)\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\ndef locate_source():\\\\\\\\\\\\\\\\n    if DATA_SOURCE is not None:\\\\\\\\\\\\\\\\n        return Path(DATA_SOURCE)\\\\\\\\\\\\\\\\n    preferred = Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/working/dcid_hf_subset/train_iphone_1_part1_20_2026_manifest_draft.csv\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    if preferred.is_file():\\\\\\\\\\\\\\\\n        return preferred\\\\\\\\\\\\\\\\n    for pattern in [\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'*_manifest_draft.csv\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'*_with_images.zip\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\']:\\\\\\\\\\\\\\\\n        found = sorted((p for base in [Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/working\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')] if base.exists() for p in base.rglob(pattern)))\\\\\\\\\\\\\\\\n        if len(found) == 1:\\\\\\\\\\\\\\\\n            return found[0]\\\\\\\\\\\\\\\\n        if len(found) > 1:\\\\\\\\\\\\\\\\n            raise RuntimeError(f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'候选不唯一，请设置 DATA_SOURCE：{found}\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    raise FileNotFoundError(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'没有找到 DCID 清单或 ZIP。请恢复下载结果，或将原图子集挂载为 Kaggle Input。\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\ndef load_official_model():\\\\\\\\\\\\\\\\n    repo = Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/working/moire_project/UHDM\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    if not (repo / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'model/nets.py\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\').is_file():\\\\\\\\\\\\\\\\n        repo = ROOT / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'UHDM\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\n        if not repo.exists():\\\\\\\\\\\\\\\\n            subprocess.check_call([\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'git\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'clone\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'--depth\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'1\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'https://github.com/CVMI-Lab/UHDM.git\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', str(repo)])\\\\\\\\\\\\\\\\n    commit = subprocess.check_output([\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'git\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'-C\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', str(repo), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'rev-parse\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'HEAD\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], text=True).strip()\\\\\\\\\\\\\\\\n    if LOCAL_WEIGHT is not None:\\\\\\\\\\\\\\\\n        weight = Path(LOCAL_WEIGHT)\\\\\\\\\\\\\\\\n        if not weight.is_file():\\\\\\\\\\\\\\\\n            raise FileNotFoundError(weight)\\\\\\\\\\\\\\\\n    else:\\\\\\\\\\\\\\\\n        known = [Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/working/moire_project/uhdm_checkpoint.pth\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), ROOT / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'uhdm_checkpoint.pth\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\']\\\\\\\\\\\\\\\\n        weight = next((p for p in known if p.is_file()), None)\\\\\\\\\\\\\\\\n        if weight is None and Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\').exists():\\\\\\\\\\\\\\\\n            found = list(Path(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'/kaggle/input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\').rglob(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'uhdm_checkpoint.pth\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'))\\\\\\\\\\\\\\\\n            if len(found) > 1:\\\\\\\\\\\\\\\\n                raise RuntimeError(f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'找到多个权重，请指定 LOCAL_WEIGHT：{found}\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n            if found:\\\\\\\\\\\\\\\\n                weight = found[0]\\\\\\\\\\\\\\\\n        if weight is None:\\\\\\\\\\\\\\\\n            import gdown\\\\\\\\\\\\\\\\n            weight = ROOT / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'uhdm_checkpoint.pth\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\n            temporary = weight.with_suffix(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'.download\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n            gdown.download(id=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'1HT_ubcAYRqzFIJ46XuPhrulJk2YFBIEl\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', output=str(temporary), quiet=False)\\\\\\\\\\\\\\\\n            if not temporary.is_file():\\\\\\\\\\\\\\\\n                raise RuntimeError(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'官方权重下载失败，请挂载旧权重并填写 LOCAL_WEIGHT\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n            temporary.replace(weight)\\\\\\\\\\\\\\\\n    spec = importlib.util.spec_from_file_location(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'dcid_esdnet\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', repo / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'model/nets.py\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    module = importlib.util.module_from_spec(spec)\\\\\\\\\\\\\\\\n    spec.loader.exec_module(module)\\\\\\\\\\\\\\\\n    cfg = yaml.safe_load((repo / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'config/uhdm_config.yaml\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\').read_text())[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'MODEL\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\']\\\\\\\\\\\\\\\\n    model = module.my_model(en_feature_num=cfg[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'EN_FEATURE_NUM\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], en_inter_num=cfg[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'EN_INTER_NUM\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], de_feature_num=cfg[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'DE_FEATURE_NUM\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], de_inter_num=cfg[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'DE_INTER_NUM\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], sam_number=cfg[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'SAM_NUMBER\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'])\\\\\\\\\\\\\\\\n    state = torch.load(weight, map_location=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'cpu\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', weights_only=True)\\\\\\\\\\\\\\\\n    if \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'state_dict\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\' in state:\\\\\\\\\\\\\\\\n        state = state[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'state_dict\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\']\\\\\\\\\\\\\\\\n    model.load_state_dict({k.removeprefix(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'module.\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'): v for k, v in state.items()}, strict=True)\\\\\\\\\\\\\\\\n    return (model.cuda().eval().requires_grad_(False), {\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'repo_commit\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': commit, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'weight_sha256\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': sha_file(weight), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'weight_path\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': str(weight)})\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\n@torch.inference_mode()\\\\\\\\\\\\\\\\ndef predict(model, array, scale):\\\\\\\\\\\\\\\\n    h, w = array.shape[:2]\\\\\\\\\\\\\\\\n    x = torch.from_numpy(array.copy()).permute(2, 0, 1)[None].float().cuda() / 255\\\\\\\\\\\\\\\\n    sh, sw = (round(h * scale), round(w * scale))\\\\\\\\\\\\\\\\n    if (sh, sw) != (h, w):\\\\\\\\\\\\\\\\n        x = F.interpolate(x, size=(sh, sw), mode=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'bilinear\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', align_corners=False, antialias=True)\\\\\\\\\\\\\\\\n    x = F.pad(x, (0, -sw % 32, 0, -sh % 32), mode=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'replicate\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    y = model(x)\\\\\\\\\\\\\\\\n    y = y[0] if isinstance(y, (tuple, list)) else y\\\\\\\\\\\\\\\\n    y = y[:, :, :sh, :sw]\\\\\\\\\\\\\\\\n    if (sh, sw) != (h, w):\\\\\\\\\\\\\\\\n        y = F.interpolate(y, size=(h, w), mode=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'bilinear\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', align_corners=False)\\\\\\\\\\\\\\\\n    return y.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\ndef metrics(pred, gt):\\\\\\\\\\\\\\\\n    pred = np.clip(pred, 0, 1).astype(np.float32)\\\\\\\\\\\\\\\\n    gt = gt.astype(np.float32)\\\\\\\\\\\\\\\\n    mse = float(np.mean((pred.astype(np.float64) - gt.astype(np.float64)) ** 2))\\\\\\\\\\\\\\\\n    return {\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'psnr\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': float(-10 * np.log10(max(mse, 1e-12))), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'ssim\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': float(structural_similarity(gt, pred, data_range=1.0, channel_axis=-1, gaussian_weights=True, sigma=1.5, use_sample_covariance=False))}\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\ndef preview(images, path):\\\\\\\\\\\\\\\\n    sheet = Image.new(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'RGB\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', (320 * len(images), 350), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'white\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    draw = ImageDraw.Draw(sheet)\\\\\\\\\\\\\\\\n    for i, (label, array) in enumerate(images):\\\\\\\\\\\\\\\\n        im = Image.fromarray(np.round(np.clip(array, 0, 1) * 255).astype(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'uint8\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'))\\\\\\\\\\\\\\\\n        sheet.paste(ImageOps.contain(im, (315, 315)), (i * 320, 30))\\\\\\\\\\\\\\\\n        draw.text((i * 320 + 5, 5), label, fill=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'black\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    sheet.save(path, quality=95)\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\ndef main():\\\\\\\\\\\\\\\\n    if not torch.cuda.is_available():\\\\\\\\\\\\\\\\n        raise RuntimeError(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'当前没有 GPU。先保存下载结果；GPU 空闲后，在 Notebook 设置中启用 GPU 再运行。\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    ROOT.mkdir(parents=True, exist_ok=True)\\\\\\\\\\\\\\\\n    source = PairSource(locate_source())\\\\\\\\\\\\\\\\n    rows = source.rows[:MAX_PAIRS]\\\\\\\\\\\\\\\\n    fingerprints = []\\\\\\\\\\\\\\\\n    for row in tqdm(rows, desc=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'检查配对\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'):\\\\\\\\\\\\\\\\n        a, g = (source.read(row, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), source.read(row, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'target\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'))\\\\\\\\\\\\\\\\n        if a.shape != g.shape or min(a.shape[:2]) < REGION:\\\\\\\\\\\\\\\\n            raise RuntimeError(f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'配对尺寸不适用：{row[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\']} {a.shape} {g.shape}\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        fingerprints.append({\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': row[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], **{k + \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'_sha256\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': hashlib.sha256(source.read_bytes(row, k)).hexdigest() for k in [\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'target\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\']}})\\\\\\\\\\\\\\\\n    out = ROOT / datetime.now().strftime(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'center768_%Y%m%d_%H%M%S_%f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    out.mkdir()\\\\\\\\\\\\\\\\n    model, metadata = load_official_model()\\\\\\\\\\\\\\\\n    protocol = {**metadata, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'data_source\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': str(source.path), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'manifest_sha256\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': hashlib.sha256(source.manifest_bytes).hexdigest(), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'fingerprints\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': fingerprints, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'region\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': REGION, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'crop\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'native-resolution center; identical for all methods\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'scales\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': SCALES, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'pairs\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': len(rows), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'torch\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': torch.__version__, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'gpu\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': torch.cuda.get_device_name(0), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'training\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': False, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'status\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'exploratory target training-pool subset; not official test or full-image score\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'ssim\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'RGB, gaussian sigma1.5, population covariance, data_range1\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'timing\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'synchronized predict call incl CPU/GPU transfer; no speedup claim; first image includes warmup overhead\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'}\\\\\\\\\\\\\\\\n    (out / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'protocol.json\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\').write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'utf-8\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    pd.DataFrame(rows).to_csv(out / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'evaluated_manifest.csv\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', index=False)\\\\\\\\\\\\\\\\n    results = []\\\\\\\\\\\\\\\\n    for index, row in enumerate(tqdm(rows, desc=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'DCID 基线推理\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')):\\\\\\\\\\\\\\\\n        a, g = (source.read(row, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), source.read(row, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'target\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'))\\\\\\\\\\\\\\\\n        h, w = a.shape[:2]\\\\\\\\\\\\\\\\n        top, left = ((h - REGION) // 2, (w - REGION) // 2)\\\\\\\\\\\\\\\\n        a = a[top:top + REGION, left:left + REGION].copy()\\\\\\\\\\\\\\\\n        gt = g[top:top + REGION, left:left + REGION].astype(np.float32) / 255\\\\\\\\\\\\\\\\n        x = a.astype(np.float32) / 255\\\\\\\\\\\\\\\\n        images = [(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'Input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', x)]\\\\\\\\\\\\\\\\n        folder = out / f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'{index:02d}\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\n        folder.mkdir()\\\\\\\\\\\\\\\\n        Image.fromarray(a).save(folder / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input.png\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        Image.fromarray(g[top:top + REGION, left:left + REGION]).save(folder / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'gt.png\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        base = {\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': row[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'index\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': index, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'top\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': top, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'left\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': left, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'region\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': REGION}\\\\\\\\\\\\\\\\n        results.append({**base, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'method\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', **metrics(x, gt), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'seconds\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': 0.0})\\\\\\\\\\\\\\\\n        for scale in SCALES:\\\\\\\\\\\\\\\\n            torch.cuda.synchronize()\\\\\\\\\\\\\\\\n            start = time.perf_counter()\\\\\\\\\\\\\\\\n            try:\\\\\\\\\\\\\\\\n                pred = predict(model, a, scale)\\\\\\\\\\\\\\\\n            except torch.cuda.OutOfMemoryError:\\\\\\\\\\\\\\\\n                torch.cuda.empty_cache()\\\\\\\\\\\\\\\\n                raise RuntimeError(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'显存不足：当前实验停止，不混合不同区域的成绩。将 REGION 改成 512 后整轮重跑；结果目录会独立保存。\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\') from None\\\\\\\\\\\\\\\\n            torch.cuda.synchronize()\\\\\\\\\\\\\\\\n            seconds = time.perf_counter() - start\\\\\\\\\\\\\\\\n            method = f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'esd_scale_{scale:g}\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\n            results.append({**base, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'method\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': method, **metrics(pred, gt), \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'seconds\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\': seconds})\\\\\\\\\\\\\\\\n            Image.fromarray(np.round(pred * 255).astype(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'uint8\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')).save(folder / f\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'{method}.png\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n            images.append((method, pred))\\\\\\\\\\\\\\\\n        preview(images + [(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'GT\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', gt)], folder / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'comparison.jpg\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        pd.DataFrame(results).to_csv(out / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'per_image.csv\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', index=False)\\\\\\\\\\\\\\\\n    frame = pd.DataFrame(results)\\\\\\\\\\\\\\\\n    summary = frame.groupby(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'method\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\').agg(psnr=(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'psnr\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'mean\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), ssim=(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'ssim\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'mean\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'), images=(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'index\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'nunique\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'))\\\\\\\\\\\\\\\\n    summary.to_csv(out / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'summary.csv\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    pivot = frame.pivot(index=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', columns=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'method\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', values=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'psnr\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    delta = pivot.sub(pivot[\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'], axis=0).drop(columns=\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    delta.to_csv(out / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'gain_vs_input.csv\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    display(summary)\\\\\\\\\\\\\\\\n    print(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'相对输入的逐图 PSNR 增益统计：\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    display(delta.agg([\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'mean\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'median\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'min\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'max\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\']))\\\\\\\\\\\\\\\\n    print(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'正增益图片数：\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    print((delta > 0).sum())\\\\\\\\\\\\\\\\n    with Image.open(out / \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'00/comparison.jpg\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\') as im:\\\\\\\\\\\\\\\\n        display(im.copy())\\\\\\\\\\\\\\\\n    bundle = out.parent / (out.name + \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'_results.zip\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    with zipfile.ZipFile(bundle, \\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'w\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', zipfile.ZIP_DEFLATED) as z:\\\\\\\\\\\\\\\\n        for p in out.rglob(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'*\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'):\\\\\\\\\\\\\\\\n            if p.is_file():\\\\\\\\\\\\\\\\n                z.write(p, p.relative_to(out))\\\\\\\\\\\\\\\\n    print(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'完成。当前仅为中心区域开发诊断；不训练、不使用正式测试集、不宣称实际加速。\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n    print(\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\'下载并上传此 ZIP：\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', bundle)\\\\\\\\\\\\\\\\n    display(FileLink(str(bundle)))\\\\\\\\\\\\\\\\n    return out\\\\\\\\\\\\\\\', H.__dict__)\\\\\\\\nfrom pathlib import Path\\\\\\\\nimport io, csv, json, random, hashlib, zipfile, time, copy\\\\\\\\nfrom datetime import datetime\\\\\\\\nimport numpy as np\\\\\\\\nimport pandas as pd\\\\\\\\nfrom PIL import Image\\\\\\\\nimport torch\\\\\\\\nimport torch.nn as nn\\\\\\\\nimport torch.nn.functional as F\\\\\\\\nfrom tqdm.auto import tqdm\\\\\\\\nfrom IPython.display import display, FileLink\\\\\\\\nDCID_SOURCE = None\\\\\\\\nFUSION_ZIP = None\\\\\\\\nWEIGHT = None\\\\\\\\nUHDM_ROOT = None\\\\\\\\nSTEPS = 300\\\\\\\\nEVAL_EVERY = 100\\\\\\\\nDCID_TRAIN = 12\\\\\\\\nSOURCE_REPLAY = 20\\\\\\\\nREGION = 768\\\\\\\\nCROP = 256\\\\\\\\nBATCH = 2\\\\\\\\nLR = 1e-05\\\\\\\\nSEED = 2026\\\\\\\\nRUN_ROOT = Path(\\\\\\\\\\\\\\\'/kaggle/working/joint_moire_runs\\\\\\\\\\\\\\\')\\\\\\\\nexec("def fusion_lowpass(x):\\\\\\\\\\\\\\\\n    v = torch.arange(-4, 5, device=x.device, dtype=x.dtype)\\\\\\\\\\\\\\\\n    k = torch.exp(-v.square() / 8)\\\\\\\\\\\\\\\\n    k = k / k.sum()\\\\\\\\\\\\\\\\n    c = x.shape[1]\\\\\\\\\\\\\\\\n    a = F.conv2d(F.pad(x, (4, 4, 0, 0), mode=\\\\\\\\\\\\\\\'replicate\\\\\\\\\\\\\\\'), k.view(1, 1, 1, 9).repeat(c, 1, 1, 1), groups=c)\\\\\\\\\\\\\\\\n    return F.conv2d(F.pad(a, (0, 0, 4, 4), mode=\\\\\\\\\\\\\\\'replicate\\\\\\\\\\\\\\\'), k.view(1, 1, 9, 1).repeat(c, 1, 1, 1), groups=c)\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\nclass ScaleFusion(nn.Module):\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\n    def __init__(self, mode=\\\\\\\\\\\\\\\'band\\\\\\\\\\\\\\\', preferred=0):\\\\\\\\\\\\\\\\n        super().__init__()\\\\\\\\\\\\\\\\n        assert mode in (\\\\\\\\\\\\\\\'spatial\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'band\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        self.mode = mode\\\\\\\\\\\\\\\\n        outputs = 6 if mode == \\\\\\\\\\\\\\\'band\\\\\\\\\\\\\\\' else 3\\\\\\\\\\\\\\\\n        self.net = nn.Sequential(nn.Conv2d(12, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, outputs, 1))\\\\\\\\\\\\\\\\n        nn.init.zeros_(self.net[-1].weight)\\\\\\\\\\\\\\\\n        nn.init.zeros_(self.net[-1].bias)\\\\\\\\\\\\\\\\n        with torch.no_grad():\\\\\\\\\\\\\\\\n            self.net[-1].bias[preferred] = 6.0\\\\\\\\\\\\\\\\n            if mode == \\\\\\\\\\\\\\\'band\\\\\\\\\\\\\\\':\\\\\\\\\\\\\\\\n                self.net[-1].bias[preferred + 3] = 6.0\\\\\\\\\\\\\\\\n\\\\\\\\\\\\\\\\n    def forward(self, x, candidates, lows):\\\\\\\\\\\\\\\\n        b, _, _, h, w = candidates.shape\\\\\\\\\\\\\\\\n        features = torch.cat((x, candidates.reshape(b, 9, h, w)), dim=1)\\\\\\\\\\\\\\\\n        features = F.interpolate(features, size=(max(1, h // 4), max(1, w // 4)), mode=\\\\\\\\\\\\\\\'area\\\\\\\\\\\\\\\')\\\\\\\\\\\\\\\\n        logits = F.interpolate(self.net(features), size=(h, w), mode=\\\\\\\\\\\\\\\'bilinear\\\\\\\\\\\\\\\', align_corners=False)\\\\\\\\\\\\\\\\n        wl = logits[:, :3].softmax(dim=1).unsqueeze(2)\\\\\\\\\\\\\\\\n        wh = logits[:, 3:].softmax(dim=1).unsqueeze(2) if self.mode == \\\\\\\\\\\\\\\'band\\\\\\\\\\\\\\\' else wl\\\\\\\\\\\\\\\\n        return (wl * lows + wh * (candidates - lows)).sum(dim=1)", globals())\\\\\\\\n\\\\\\\\ndef find_unique(pattern, explicit):\\\\\\\\n    if explicit is not None:\\\\\\\\n        p = Path(explicit)\\\\\\\\n        if not p.is_file():\\\\\\\\n            raise FileNotFoundError(p)\\\\\\\\n        return p\\\\\\\\n    found = sorted((p for root in [Path(\\\\\\\\\\\\\\\'/kaggle/input\\\\\\\\\\\\\\\'), Path(\\\\\\\\\\\\\\\'/kaggle/working\\\\\\\\\\\\\\\')] if root.exists() for p in root.rglob(pattern)))\\\\\\\\n    if len(found) != 1:\\\\\\\\n        raise RuntimeError(f\\\\\\\\\\\\\\\'请在配置中填写 {pattern} 的路径，候选={found}\\\\\\\\\\\\\\\')\\\\\\\\n    return found[0]\\\\\\\\n\\\\\\\\ndef load_old_fusion():\\\\\\\\n    if FUSION_ZIP is not None and Path(FUSION_ZIP).is_dir():\\\\\\\\n        folder = Path(FUSION_ZIP)\\\\\\\\n    else:\\\\\\\\n        found = [Path(FUSION_ZIP)] if FUSION_ZIP is not None else [p for root in [Path(\\\\\\\\\\\\\\\'/kaggle/input\\\\\\\\\\\\\\\'), Path(\\\\\\\\\\\\\\\'/kaggle/working\\\\\\\\\\\\\\\')] if root.exists() for p in root.rglob(\\\\\\\\\\\\\\\'fusion_training_results.zip\\\\\\\\\\\\\\\')]\\\\\\\\n        if len(found) > 1:\\\\\\\\n            raise RuntimeError(f\\\\\\\\\\\\\\\'旧融合包不唯一，请填写 FUSION_ZIP：{found}\\\\\\\\\\\\\\\')\\\\\\\\n        if found:\\\\\\\\n            with zipfile.ZipFile(found[0]) as z:\\\\\\\\n                p = z.read(\\\\\\\\\\\\\\\'protocol.json\\\\\\\\\\\\\\\')\\\\\\\\n                ck = z.read(\\\\\\\\\\\\\\\'band/best.pt\\\\\\\\\\\\\\\')\\\\\\\\n            return (json.loads(p), torch.load(io.BytesIO(ck), map_location=\\\\\\\\\\\\\\\'cpu\\\\\\\\\\\\\\\', weights_only=True), hashlib.sha256(p + ck).hexdigest())\\\\\\\\n        folders = sorted({p.parent.parent for root in [Path(\\\\\\\\\\\\\\\'/kaggle/input\\\\\\\\\\\\\\\'), Path(\\\\\\\\\\\\\\\'/kaggle/working\\\\\\\\\\\\\\\')] if root.exists() for p in root.rglob(\\\\\\\\\\\\\\\'best.pt\\\\\\\\\\\\\\\') if p.parent.name == \\\\\\\\\\\\\\\'band\\\\\\\\\\\\\\\' and (p.parent.parent / \\\\\\\\\\\\\\\'protocol.json\\\\\\\\\\\\\\\').is_file()})\\\\\\\\n        if len(folders) != 1:\\\\\\\\n            raise RuntimeError(f\\\\\\\\\\\\\\\'请挂载旧融合结果包，FUSION_ZIP 可填写 ZIP 路径或展开后的根目录。候选={folders}\\\\\\\\\\\\\\\')\\\\\\\\n        folder = folders[0]\\\\\\\\n    p = (folder / \\\\\\\\\\\\\\\'protocol.json\\\\\\\\\\\\\\\').read_bytes()\\\\\\\\n    ck = (folder / \\\\\\\\\\\\\\\'band/best.pt\\\\\\\\\\\\\\\').read_bytes()\\\\\\\\n    return (json.loads(p), torch.load(io.BytesIO(ck), map_location=\\\\\\\\\\\\\\\'cpu\\\\\\\\\\\\\\\', weights_only=True), hashlib.sha256(p + ck).hexdigest())\\\\\\\\n\\\\\\\\ndef source_path(old, root):\\\\\\\\n    p = Path(old)\\\\\\\\n    if p.is_file():\\\\\\\\n        return p\\\\\\\\n    tail = old.replace(\\\\\\\\\\\\\\\'\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'/\\\\\\\\\\\\\\\').split(\\\\\\\\\\\\\\\'/uhdm-dataset/\\\\\\\\\\\\\\\', 1)[-1]\\\\\\\\n    direct = root / tail\\\\\\\\n    if direct.is_file():\\\\\\\\n        return direct\\\\\\\\n    matches = [p for p in root.rglob(Path(tail).name) if p.parent.name == Path(tail).parent.name]\\\\\\\\n    if len(matches) != 1:\\\\\\\\n        raise RuntimeError(f\\\\\\\\\\\\\\\'UHDM 图片无法唯一映射：{old}; {matches}\\\\\\\\\\\\\\\')\\\\\\\\n    return matches[0]\\\\\\\\n\\\\\\\\ndef crop_pair(a, g, label):\\\\\\\\n    if a.shape != g.shape or min(a.shape[:2]) < REGION:\\\\\\\\n        raise RuntimeError(f\\\\\\\\\\\\\\\'尺寸异常：{label}\\\\\\\\\\\\\\\')\\\\\\\\n    h, w = a.shape[:2]\\\\\\\\n    t, l = ((h - REGION) // 2, (w - REGION) // 2)\\\\\\\\n    return (a[t:t + REGION, l:l + REGION].copy(), g[t:t + REGION, l:l + REGION].copy())\\\\\\\\n\\\\\\\\ndef load_source(items, root, split):\\\\\\\\n    output = []\\\\\\\\n    for row in tqdm(items, desc=\\\\\\\\\\\\\\\'UHDM \\\\\\\\\\\\\\\' + split):\\\\\\\\n        xp, gp = (source_path(row[\\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\'], root), source_path(row[\\\\\\\\\\\\\\\'gt\\\\\\\\\\\\\\\'], root))\\\\\\\\n        with Image.open(xp) as im:\\\\\\\\n            a = np.array(im.convert(\\\\\\\\\\\\\\\'RGB\\\\\\\\\\\\\\\'))\\\\\\\\n        with Image.open(gp) as im:\\\\\\\\n            g = np.array(im.convert(\\\\\\\\\\\\\\\'RGB\\\\\\\\\\\\\\\'))\\\\\\\\n        a, g = crop_pair(a, g, str(gp))\\\\\\\\n        output.append({\\\\\\\\\\\\\\\'id\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\'UHDM/\\\\\\\\\\\\\\\' + gp.parent.name + \\\\\\\\\\\\\\\'/\\\\\\\\\\\\\\\' + gp.stem, \\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\': a, \\\\\\\\\\\\\\\'gt\\\\\\\\\\\\\\\': g, \\\\\\\\\\\\\\\'gt_hash\\\\\\\\\\\\\\\': H.sha_file(gp), \\\\\\\\\\\\\\\'input_hash\\\\\\\\\\\\\\\': H.sha_file(xp), \\\\\\\\\\\\\\\'path\\\\\\\\\\\\\\\': str(gp), \\\\\\\\\\\\\\\'split\\\\\\\\\\\\\\\': split, \\\\\\\\\\\\\\\'group\\\\\\\\\\\\\\\': str(row[\\\\\\\\\\\\\\\'group\\\\\\\\\\\\\\\'])})\\\\\\\\n    return output\\\\\\\\n\\\\\\\\ndef tensor(a):\\\\\\\\n    return torch.from_numpy(a.copy()).permute(2, 0, 1)[None].float().cuda() / 255\\\\\\\\n\\\\\\\\ndef anchor(y, x):\\\\\\\\n    return y + (x - y).mean(dim=(-2, -1), keepdim=True)\\\\\\\\n\\\\\\\\n@torch.inference_mode()\\\\\\\\ndef forward_native(model, a):\\\\\\\\n    x = tensor(a)\\\\\\\\n    h, w = x.shape[-2:]\\\\\\\\n    y = model(F.pad(x, (0, -w % 32, 0, -h % 32), mode=\\\\\\\\\\\\\\\'replicate\\\\\\\\\\\\\\\'))[0][:, :, :h, :w]\\\\\\\\n    return (y, x)\\\\\\\\n\\\\\\\\ndef arr(t):\\\\\\\\n    return t[0].detach().clamp(0, 1).permute(1, 2, 0).cpu().numpy()\\\\\\\\n\\\\\\\\n@torch.inference_mode()\\\\\\\\ndef evaluate(model, items, label, domain, out, step, anchored=False, include_baselines=False, fusion=None):\\\\\\\\n    model.eval()\\\\\\\\n    records = []\\\\\\\\n    for i, p in enumerate(items):\\\\\\\\n        raw, x = forward_native(model, p[\\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\'])\\\\\\\\n        g = p[\\\\\\\\\\\\\\\'gt\\\\\\\\\\\\\\\'].astype(np.float32) / 255\\\\\\\\n        predictions = {label: arr(anchor(raw, x) if anchored else raw)}\\\\\\\\n        if include_baselines:\\\\\\\\n            predictions = {\\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\': p[\\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\'].astype(np.float32) / 255, \\\\\\\\\\\\\\\'original\\\\\\\\\\\\\\\': arr(raw), \\\\\\\\\\\\\\\'mean_anchor\\\\\\\\\\\\\\\': arr(anchor(raw.clamp(0, 1), x))}\\\\\\\\n            if fusion is not None:\\\\\\\\n                candidates = torch.stack([torch.from_numpy(H.predict(model, p[\\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\'], s)).permute(2, 0, 1).cuda() for s in (0.5, 0.75, 1.0)])[None]\\\\\\\\n                lows = fusion_lowpass(candidates[0])[None]\\\\\\\\n                predictions[\\\\\\\\\\\\\\\'old_band\\\\\\\\\\\\\\\'] = arr(fusion(x, candidates, lows))\\\\\\\\n        for method, pred in predictions.items():\\\\\\\\n            score = H.metrics(pred, g)\\\\\\\\n            records.append({\\\\\\\\\\\\\\\'domain\\\\\\\\\\\\\\\': domain, \\\\\\\\\\\\\\\'id\\\\\\\\\\\\\\\': p[\\\\\\\\\\\\\\\'id\\\\\\\\\\\\\\\'], \\\\\\\\\\\\\\\'method\\\\\\\\\\\\\\\': method, \\\\\\\\\\\\\\\'step\\\\\\\\\\\\\\\': step, **score})\\\\\\\\n            dest = out / \\\\\\\\\\\\\\\'images\\\\\\\\\\\\\\\' / domain / method\\\\\\\\n            dest.mkdir(parents=True, exist_ok=True)\\\\\\\\n            Image.fromarray(np.round(pred * 255).astype(\\\\\\\\\\\\\\\'uint8\\\\\\\\\\\\\\\')).save(dest / f\\\\\\\\\\\\\\\'{i:02d}.png\\\\\\\\\\\\\\\')\\\\\\\\n        if i < 3:\\\\\\\\n            dest = out / \\\\\\\\\\\\\\\'previews\\\\\\\\\\\\\\\'\\\\\\\\n            dest.mkdir(exist_ok=True)\\\\\\\\n            H.preview([(\\\\\\\\\\\\\\\'Input\\\\\\\\\\\\\\\', p[\\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\'] / 255)] + list(predictions.items()) + [(\\\\\\\\\\\\\\\'GT\\\\\\\\\\\\\\\', g)], dest / f\\\\\\\\\\\\\\\'{domain}_{label}_{step}_{i}.jpg\\\\\\\\\\\\\\\')\\\\\\\\n    return records\\\\\\\\n\\\\\\\\ndef check_split(training, validation):\\\\\\\\n    for field in [\\\\\\\\\\\\\\\'gt_hash\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'input_hash\\\\\\\\\\\\\\\']:\\\\\\\\n        overlap = {p[field] for p in training} & {p[field] for p in validation}\\\\\\\\n        if overlap:\\\\\\\\n            raise RuntimeError(f\\\\\\\\\\\\\\\'训练/验证有精确重复 {field}；请修复划分，尚未启动训练\\\\\\\\\\\\\\\')\\\\\\\\n\\\\\\\\ndef draw_batch(items, rng):\\\\\\\\n    xs = []\\\\\\\\n    gs = []\\\\\\\\n    for _ in range(BATCH):\\\\\\\\n        p = rng.choice(items)\\\\\\\\n        t = rng.randrange(REGION - CROP + 1)\\\\\\\\n        l = rng.randrange(REGION - CROP + 1)\\\\\\\\n        a = p[\\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\'][t:t + CROP, l:l + CROP]\\\\\\\\n        g = p[\\\\\\\\\\\\\\\'gt\\\\\\\\\\\\\\\'][t:t + CROP, l:l + CROP]\\\\\\\\n        if rng.random() < 0.5:\\\\\\\\n            a, g = (np.flip(a, 1), np.flip(g, 1))\\\\\\\\n        xs.append(tensor(a))\\\\\\\\n        gs.append(tensor(g))\\\\\\\\n    return (torch.cat(xs), torch.cat(gs))\\\\\\\\n\\\\\\\\ndef save_bundle(out):\\\\\\\\n    bundle = out.parent / (out.name + \\\\\\\\\\\\\\\'_results.zip\\\\\\\\\\\\\\\')\\\\\\\\n    with zipfile.ZipFile(bundle, \\\\\\\\\\\\\\\'w\\\\\\\\\\\\\\\', zipfile.ZIP_DEFLATED) as z:\\\\\\\\n        for p in out.rglob(\\\\\\\\\\\\\\\'*\\\\\\\\\\\\\\\'):\\\\\\\\n            if p.is_file():\\\\\\\\n                z.write(p, p.relative_to(out))\\\\\\\\n    print(\\\\\\\\\\\\\\\'结果包：\\\\\\\\\\\\\\\', bundle)\\\\\\\\n    display(FileLink(str(bundle)))\\\\\\\\n\\\\\\\\ndef run_joint():\\\\\\\\n    assert torch.cuda.is_available(), \\\\\\\\\\\\\\\'需要 GPU；CPU 会话先保存数据即可\\\\\\\\\\\\\\\'\\\\\\\\n    assert REGION % 32 == 0 and CROP % 32 == 0 and (CROP <= REGION)\\\\\\\\n    protocol, checkpoint, old_fusion_hash = load_old_fusion()\\\\\\\\n    if UHDM_ROOT is None:\\\\\\\\n        roots = sorted((p for p in Path(\\\\\\\\\\\\\\\'/kaggle/input\\\\\\\\\\\\\\\').rglob(\\\\\\\\\\\\\\\'uhdm-dataset\\\\\\\\\\\\\\\') if p.is_dir()))\\\\\\\\n        if len(roots) != 1:\\\\\\\\n            raise RuntimeError(\\\\\\\\\\\\\\\'请 Add Input 挂载 soumikrakshit/uhdm-dataset，或填写 UHDM_ROOT\\\\\\\\\\\\\\\')\\\\\\\\n        source_root = roots[0]\\\\\\\\n    else:\\\\\\\\n        source_root = Path(UHDM_ROOT)\\\\\\\\n    H.DATA_SOURCE = DCID_SOURCE\\\\\\\\n    source = H.PairSource(H.locate_source())\\\\\\\\n    rows = sorted(source.rows, key=lambda r: r[\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\'])\\\\\\\\n    random.Random(SEED).shuffle(rows)\\\\\\\\n    assert len(rows) == 20, \\\\\\\\\\\\\\\'首轮固定使用已检查的 20 对，不自动加入其他图片\\\\\\\\\\\\\\\'\\\\\\\\n    target = []\\\\\\\\n    for i, row in enumerate(tqdm(rows, desc=\\\\\\\\\\\\\\\'DCID 配对\\\\\\\\\\\\\\\')):\\\\\\\\n        a, g = crop_pair(source.read(row, \\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\'), source.read(row, \\\\\\\\\\\\\\\'target\\\\\\\\\\\\\\\'), row[\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\'])\\\\\\\\n        target.append({\\\\\\\\\\\\\\\'id\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\'DCID/\\\\\\\\\\\\\\\' + row[\\\\\\\\\\\\\\\'sample_id\\\\\\\\\\\\\\\'], \\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\': a, \\\\\\\\\\\\\\\'gt\\\\\\\\\\\\\\\': g, \\\\\\\\\\\\\\\'gt_hash\\\\\\\\\\\\\\\': hashlib.sha256(source.read_bytes(row, \\\\\\\\\\\\\\\'target\\\\\\\\\\\\\\\')).hexdigest(), \\\\\\\\\\\\\\\'input_hash\\\\\\\\\\\\\\\': hashlib.sha256(source.read_bytes(row, \\\\\\\\\\\\\\\'input\\\\\\\\\\\\\\\')).hexdigest(), \\\\\\\\\\\\\\\'split\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\'train\\\\\\\\\\\\\\\' if i < DCID_TRAIN else \\\\\\\\\\\\\\\'development_val\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'path\\\\\\\\\\\\\\\': row[\\\\\\\\\\\\\\\'target\\\\\\\\\\\\\\\']})\\\\\\\\n    target_train, target_val = (target[:DCID_TRAIN], target[DCID_TRAIN:])\\\\\\\\n    source_train = load_source(protocol[\\\\\\\\\\\\\\\'manifest\\\\\\\\\\\\\\\'][\\\\\\\\\\\\\\\'train\\\\\\\\\\\\\\\'][:SOURCE_REPLAY], source_root, \\\\\\\\\\\\\\\'replay_train\\\\\\\\\\\\\\\')\\\\\\\\n    source_val = load_source(protocol[\\\\\\\\\\\\\\\'manifest\\\\\\\\\\\\\\\'][\\\\\\\\\\\\\\\'val\\\\\\\\\\\\\\\'], source_root, \\\\\\\\\\\\\\\'old_development_val\\\\\\\\\\\\\\\')\\\\\\\\n    check_split(source_train + target_train, source_val + target_val)\\\\\\\\n    if {p[\\\\\\\\\\\\\\\'group\\\\\\\\\\\\\\\'] for p in source_train} & {p[\\\\\\\\\\\\\\\'group\\\\\\\\\\\\\\\'] for p in source_val}:\\\\\\\\n        raise RuntimeError(\\\\\\\\\\\\\\\'UHDM 旧组划分有交集\\\\\\\\\\\\\\\')\\\\\\\\n    out = RUN_ROOT / datetime.now().strftime(\\\\\\\\\\\\\\\'joint_%Y%m%d_%H%M%S\\\\\\\\\\\\\\\')\\\\\\\\n    out.mkdir(parents=True)\\\\\\\\n    H.ROOT = RUN_ROOT / \\\\\\\\\\\\\\\'official\\\\\\\\\\\\\\\'\\\\\\\\n    H.ROOT.mkdir(exist_ok=True)\\\\\\\\n    H.LOCAL_WEIGHT = WEIGHT\\\\\\\\n    model, metadata = H.load_official_model()\\\\\\\\n    digest = hashlib.sha256()\\\\\\\\n    for k, v in sorted(model.state_dict().items()):\\\\\\\\n        digest.update(k.encode())\\\\\\\\n        digest.update(v.detach().cpu().contiguous().numpy().tobytes())\\\\\\\\n    if digest.hexdigest() != protocol[\\\\\\\\\\\\\\\'weight_sha256\\\\\\\\\\\\\\\']:\\\\\\\\n        raise RuntimeError(\\\\\\\\\\\\\\\'原始主干与旧融合实验权重不一致，停止，不能混接\\\\\\\\\\\\\\\')\\\\\\\\n    band = ScaleFusion(\\\\\\\\\\\\\\\'band\\\\\\\\\\\\\\\', int(checkpoint[\\\\\\\\\\\\\\\'settings\\\\\\\\\\\\\\\'][\\\\\\\\\\\\\\\'preferred\\\\\\\\\\\\\\\'])).cuda().eval()\\\\\\\\n    band.load_state_dict(checkpoint[\\\\\\\\\\\\\\\'model\\\\\\\\\\\\\\\'], strict=True)\\\\\\\\n    for p in band.parameters():\\\\\\\\n        p.requires_grad_(False)\\\\\\\\n    manifest = [{k: v for k, v in p.items() if k not in [\\\\\\\\\\\\\\\'x\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'gt\\\\\\\\\\\\\\\']} for p in target + source_train + source_val]\\\\\\\\n    pd.DataFrame(manifest).to_csv(out / \\\\\\\\\\\\\\\'manifest.csv\\\\\\\\\\\\\\\', index=False)\\\\\\\\n    settings = {**metadata, \\\\\\\\\\\\\\\'seed\\\\\\\\\\\\\\\': SEED, \\\\\\\\\\\\\\\'region\\\\\\\\\\\\\\\': REGION, \\\\\\\\\\\\\\\'crop\\\\\\\\\\\\\\\': CROP, \\\\\\\\\\\\\\\'batch\\\\\\\\\\\\\\\': BATCH, \\\\\\\\\\\\\\\'steps_per_branch\\\\\\\\\\\\\\\': STEPS, \\\\\\\\\\\\\\\'lr\\\\\\\\\\\\\\\': LR, \\\\\\\\\\\\\\\'source_replay_images\\\\\\\\\\\\\\\': len(source_train), \\\\\\\\\\\\\\\'source_val_images\\\\\\\\\\\\\\\': len(source_val), \\\\\\\\\\\\\\\'target_train_images\\\\\\\\\\\\\\\': len(target_train), \\\\\\\\\\\\\\\'target_val_images\\\\\\\\\\\\\\\': len(target_val), \\\\\\\\\\\\\\\'replay_schedule\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\'three DCID batches then one UHDM batch; total optimizer steps equal\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'trainable\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\'decoder only; encoder frozen eval; MSE on unclipped input-mean-anchored prediction; adaptation eval anchors raw output then clips; fixed mean_anchor baseline anchors clipped original prediction\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'old_fusion_protocol_and_band_sha256\\\\\\\\\\\\\\\': old_fusion_hash, \\\\\\\\\\\\\\\'old_protocol\\\\\\\\\\\\\\\': protocol, \\\\\\\\\\\\\\\'torch\\\\\\\\\\\\\\\': torch.__version__, \\\\\\\\\\\\\\\'gpu\\\\\\\\\\\\\\\': torch.cuda.get_device_name(0), \\\\\\\\\\\\\\\'limitations\\\\\\\\\\\\\\\': \\\\\\\\\\\\\\\'DCID image split, scene grouping unknown, prior inspected data; UHDM old development val; no independent test claim. Fixed last step is primary; best target checkpoint is exploratory. Same total steps means different target update counts.\\\\\\\\\\\\\\\'}\\\\\\\\n    (out / \\\\\\\\\\\\\\\'protocol.json\\\\\\\\\\\\\\\').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding=\\\\\\\\\\\\\\\'utf-8\\\\\\\\\\\\\\\')\\\\\\\\n    initial = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\\\\\\\\n    records = []\\\\\\\\n    history = []\\\\\\\\n    try:\\\\\\\\n        print(\\\\\\\\\\\\\\\'阶段 1：两数据集固定基线与旧融合，先得到可下载结果\\\\\\\\\\\\\\\')\\\\\\\\n        for domain, items in [(\\\\\\\\\\\\\\\'DCID\\\\\\\\\\\\\\\', target_val), (\\\\\\\\\\\\\\\'UHDM\\\\\\\\\\\\\\\', source_val)]:\\\\\\\\n            records += evaluate(model, items, \\\\\\\\\\\\\\\'baseline\\\\\\\\\\\\\\\', domain, out, 0, include_baselines=True, fusion=band)\\\\\\\\n        del band\\\\\\\\n        torch.cuda.empty_cache()\\\\\\\\n\\\\\\\\n        def persist():\\\\\\\\n            frame = pd.DataFrame(records)\\\\\\\\n            frame.to_csv(out / \\\\\\\\\\\\\\\'per_image.csv\\\\\\\\\\\\\\\', index=False)\\\\\\\\n            table = frame.groupby([\\\\\\\\\\\\\\\'domain\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'method\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'step\\\\\\\\\\\\\\\']).agg(psnr=(\\\\\\\\\\\\\\\'psnr\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'mean\\\\\\\\\\\\\\\'), ssim=(\\\\\\\\\\\\\\\'ssim\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'mean\\\\\\\\\\\\\\\'), images=(\\\\\\\\\\\\\\\'id\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'nunique\\\\\\\\\\\\\\\'))\\\\\\\\n            table.to_csv(out / \\\\\\\\\\\\\\\'summary.csv\\\\\\\\\\\\\\\')\\\\\\\\n            pd.DataFrame(history).to_csv(out / \\\\\\\\\\\\\\\'history.csv\\\\\\\\\\\\\\\', index=False)\\\\\\\\n            return table\\\\\\\\n        display(persist())\\\\\\\\n        save_bundle(out)\\\\\\\\n        for branch in [\\\\\\\\\\\\\\\'target_only\\\\\\\\\\\\\\\', \\\\\\\\\\\\\\\'target_with_replay\\\\\\\\\\\\\\\']:\\\\\\\\n            model.load_state_dict(initial, strict=True)\\\\\\\\n            model.requires_grad_(False)\\\\\\\\n            model.decoder.requires_grad_(True)\\\\\\\\n            opt = torch.optim.Adam(model.decoder.parameters(), lr=LR)\\\\\\\\n            rng = random.Random(SEED)\\\\\\\\n            torch.manual_seed(SEED)\\\\\\\\n            torch.cuda.manual_seed_all(SEED)\\\\\\\\n            best = -float(\\\\\\\\\\\\\\\'inf\\\\\\\\\\\\\\\')\\\\\\\\n            begin = time.perf_counter()\\\\\\\\n            for step in tqdm(range(1, STEPS + 1), desc=branch):\\\\\\\\n                model.train()\\\\\\\\n                model.encoder.eval()\\\\\\\\n                pool = source_train if branch == \\\\\\\\\\\\\\\'target_with_replay\\\\\\\\\\\\\\\' and step % 4 == 0 else target_train\\\\\\\\n                x, g = draw_batch(pool, rng)\\\\\\\\n                opt.zero_grad(set_to_none=True)\\\\\\\\n                y = model(x)[0]\\\\\\\\n                loss = F.mse_loss(anchor(y, x), g)\\\\\\\\n                if not torch.isfinite(loss):\\\\\\\\n                    raise RuntimeError(\\\\\\\\\\\\\\\'loss 非有限值，停止\\\\\\\\\\\\\\\')\\\\\\\\n                loss.backward()\\\\\\\\n                torch.nn.utils.clip_grad_norm_(model.decoder.parameters(), 1.0)\\\\\\\\n                opt.step()\\\\\\\\n                if step == 20:\\\\\\\\n                    torch.cuda.synchronize()\\\\\\\\n                    elapsed = time.perf_counter() - begin\\\\\\\\n                    print(f\\\\\\\\\\\\\\\'{branch} 前20步 {elapsed:.1f}s；仅训练部分预计剩余 {(STEPS - step) * elapsed / 20 / 60:.1f} 分钟，另有评价开销\\\\\\\\\\\\\\\')\\\\\\\\n                if step % EVAL_EVERY == 0 or step == STEPS:\\\\\\\\n                    metrics = []\\\\\\\\n                    for domain, items in [(\\\\\\\\\\\\\\\'DCID\\\\\\\\\\\\\\\', target_val), (\\\\\\\\\\\\\\\'UHDM\\\\\\\\\\\\\\\', source_val)]:\\\\\\\\n                        metrics += evaluate(model, items, branch, domain, out, step, anchored=True)\\\\\\\\n                    records += metrics\\\\\\\\n                    score = float(np.mean([r[\\\\\\\\\\\\\\\'psnr\\\\\\\\\\\\\\\'] for r in metrics if r[\\\\\\\\\\\\\\\'domain\\\\\\\\\\\\\\\'] == \\\\\\\\\\\\\\\'DCID\\\\\\\\\\\\\\\']))\\\\\\\\n                    history.append({\\\\\\\\\\\\\\\'branch\\\\\\\\\\\\\\\': branch, \\\\\\\\\\\\\\\'step\\\\\\\\\\\\\\\': step, \\\\\\\\\\\\\\\'train_loss\\\\\\\\\\\\\\\': float(loss.detach()), \\\\\\\\\\\\\\\'target_val_psnr\\\\\\\\\\\\\\\': score, \\\\\\\\\\\\\\\'elapsed_seconds\\\\\\\\\\\\\\\': time.perf_counter() - begin})\\\\\\\\n                    payload = {\\\\\\\\\\\\\\\'decoder\\\\\\\\\\\\\\\': model.decoder.state_dict(), \\\\\\\\\\\\\\\'step\\\\\\\\\\\\\\\': step, \\\\\\\\\\\\\\\'branch\\\\\\\\\\\\\\\': branch, \\\\\\\\\\\\\\\'settings\\\\\\\\\\\\\\\': {k: v for k, v in settings.items() if k != \\\\\\\\\\\\\\\'old_protocol\\\\\\\\\\\\\\\'}}\\\\\\\\n                    torch.save(payload, out / (branch + \\\\\\\\\\\\\\\'_last.pt\\\\\\\\\\\\\\\'))\\\\\\\\n                    if score > best:\\\\\\\\n                        best = score\\\\\\\\n                        torch.save(payload, out / (branch + \\\\\\\\\\\\\\\'_best_target_dev.pt\\\\\\\\\\\\\\\'))\\\\\\\\n                    display(persist())\\\\\\\\n            del opt\\\\\\\\n            save_bundle(out)\\\\\\\\n        print(\\\\\\\\\\\\\\\'完成：优先比较两个分支固定第300步与 mean_anchor；同时查看 UHDM 是否退化。\\\\\\\\\\\\\\\')\\\\\\\\n        return out\\\\\\\\n    except Exception as error:\\\\\\\\n        (out / \\\\\\\\\\\\\\\'error.txt\\\\\\\\\\\\\\\').write_text(repr(error), encoding=\\\\\\\\\\\\\\\'utf-8\\\\\\\\\\\\\\\')\\\\\\\\n        if records:\\\\\\\\n            pd.DataFrame(records).to_csv(out / \\\\\\\\\\\\\\\'partial_per_image.csv\\\\\\\\\\\\\\\', index=False)\\\\\\\\n        save_bundle(out)\\\\\\\\n        raise\\\\\\\', J.__dict__)\\\\nfrom pathlib import Path\\\\nimport json, random, hashlib, time, zipfile\\\\nfrom datetime import datetime\\\\nimport numpy as np\\\\nimport pandas as pd\\\\nfrom PIL import Image\\\\nimport torch\\\\nfrom tqdm.auto import tqdm\\\\nfrom IPython.display import display, FileLink\\\\nDCID_SOURCE = None\\\\nFUSION_ZIP = None\\\\nUHDM_ROOT = None\\\\nWEIGHT = None\\\\nREGION = 768\\\\nSEED = 2026\\\\nRIDGE_ALPHAS = (1.0, 10.0, 100.0)\\\\nPRIMARY_ALPHA = 10.0\\\\nMAX_SHIFT = 0.25\\\\nROOT = Path(\\\\\\\'/kaggle/working/color_calibration\\\\\\\')\\\\n\\\\ndef fingerprint(model):\\\\n    digest = hashlib.sha256()\\\\n    for k, v in sorted(model.state_dict().items()):\\\\n        digest.update(k.encode())\\\\n        digest.update(v.detach().cpu().contiguous().numpy().tobytes())\\\\n    return digest.hexdigest()\\\\n\\\\ndef prepare():\\\\n    assert torch.cuda.is_available(), \\\\\\\'缓存预测需要GPU；缓存完成后的拟合与评价使用CPU\\\\\\\'\\\\n    J.REGION = REGION\\\\n    J.DCID_SOURCE = DCID_SOURCE\\\\n    J.FUSION_ZIP = FUSION_ZIP\\\\n    protocol, ck, old_hash = J.load_old_fusion()\\\\n    if UHDM_ROOT is None:\\\\n        roots = sorted((p for p in Path(\\\\\\\'/kaggle/input\\\\\\\').rglob(\\\\\\\'uhdm-dataset\\\\\\\') if p.is_dir()))\\\\n        if len(roots) != 1:\\\\n            raise RuntimeError(\\\\\\\'请挂载 soumikrakshit/uhdm-dataset，或设置 UHDM_ROOT\\\\\\\')\\\\n        root = roots[0]\\\\n    else:\\\\n        root = Path(UHDM_ROOT)\\\\n    J.H.DATA_SOURCE = DCID_SOURCE\\\\n    source = J.H.PairSource(J.H.locate_source())\\\\n    rows = sorted(source.rows, key=lambda r: r[\\\\\\\'sample_id\\\\\\\'])\\\\n    random.Random(SEED).shuffle(rows)\\\\n    assert len(rows) == 20, \\\\\\\'本轮固定复用原20对，不能悄悄更换划分\\\\\\\'\\\\n    target = []\\\\n    for i, r in enumerate(tqdm(rows, desc=\\\\\\\'读取DCID\\\\\\\')):\\\\n        a, g = J.crop_pair(source.read(r, \\\\\\\'input\\\\\\\'), source.read(r, \\\\\\\'target\\\\\\\'), r[\\\\\\\'sample_id\\\\\\\'])\\\\n        target.append({\\\\\\\'id\\\\\\\': \\\\\\\'DCID/\\\\\\\' + r[\\\\\\\'sample_id\\\\\\\'], \\\\\\\'domain\\\\\\\': \\\\\\\'DCID\\\\\\\', \\\\\\\'split\\\\\\\': \\\\\\\'train\\\\\\\' if i < 12 else \\\\\\\'val\\\\\\\', \\\\\\\'x\\\\\\\': a, \\\\\\\'gt\\\\\\\': g, \\\\\\\'gt_hash\\\\\\\': hashlib.sha256(source.read_bytes(r, \\\\\\\'target\\\\\\\')).hexdigest(), \\\\\\\'input_hash\\\\\\\': hashlib.sha256(source.read_bytes(r, \\\\\\\'input\\\\\\\')).hexdigest()})\\\\n    train = J.load_source(protocol[\\\\\\\'manifest\\\\\\\'][\\\\\\\'train\\\\\\\'][:20], root, \\\\\\\'train\\\\\\\')\\\\n    val = J.load_source(protocol[\\\\\\\'manifest\\\\\\\'][\\\\\\\'val\\\\\\\'], root, \\\\\\\'val\\\\\\\')\\\\n    for p in train + val:\\\\n        p[\\\\\\\'domain\\\\\\\'] = \\\\\\\'UHDM\\\\\\\'\\\\n    items = target + train + val\\\\n    J.check_split([p for p in items if p[\\\\\\\'split\\\\\\\'] == \\\\\\\'train\\\\\\\'], [p for p in items if p[\\\\\\\'split\\\\\\\'] == \\\\\\\'val\\\\\\\'])\\\\n    if {p[\\\\\\\'group\\\\\\\'] for p in train} & {p[\\\\\\\'group\\\\\\\'] for p in val}:\\\\n        raise RuntimeError(\\\\\\\'UHDM组冲突\\\\\\\')\\\\n    J.H.ROOT = ROOT / \\\\\\\'official\\\\\\\'\\\\n    J.H.ROOT.mkdir(parents=True, exist_ok=True)\\\\n    J.H.LOCAL_WEIGHT = WEIGHT\\\\n    if WEIGHT is None:\\\\n        known = [Path(\\\\\\\'/kaggle/working/joint_moire_runs/official/uhdm_checkpoint.pth\\\\\\\'), Path(\\\\\\\'/kaggle/working/dcid_baseline/uhdm_checkpoint.pth\\\\\\\')]\\\\n        J.H.LOCAL_WEIGHT = next((p for p in known if p.is_file()), None)\\\\n    model, meta = J.H.load_official_model()\\\\n    assert fingerprint(model) == protocol[\\\\\\\'weight_sha256\\\\\\\'], \\\\\\\'主干权重与旧融合协议不匹配\\\\\\\'\\\\n    band = J.ScaleFusion(\\\\\\\'band\\\\\\\', int(ck[\\\\\\\'settings\\\\\\\'][\\\\\\\'preferred\\\\\\\'])).cuda().eval()\\\\n    band.load_state_dict(ck[\\\\\\\'model\\\\\\\'], strict=True)\\\\n    band.requires_grad_(False)\\\\n    return (items, model, band, {**meta, \\\\\\\'old_fusion_hash\\\\\\\': old_hash})\\\\n\\\\n@torch.inference_mode()\\\\ndef cache_predictions(items, model, band, meta):\\\\n    signature = {\\\\\\\'meta\\\\\\\': meta, \\\\\\\'region\\\\\\\': REGION, \\\\\\\'protocol\\\\\\\': \\\\\\\'clipped_float16_original_and_band_v1\\\\\\\', \\\\\\\'samples\\\\\\\': [{k: p[k] for k in [\\\\\\\'id\\\\\\\', \\\\\\\'split\\\\\\\', \\\\\\\'domain\\\\\\\', \\\\\\\'input_hash\\\\\\\', \\\\\\\'gt_hash\\\\\\\']} for p in items]}\\\\n    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]\\\\n    folder = ROOT / \\\\\\\'cache\\\\\\\' / key\\\\n    folder.mkdir(parents=True, exist_ok=True)\\\\n    (folder / \\\\\\\'signature.json\\\\\\\').write_text(json.dumps(signature, indent=2), encoding=\\\\\\\'utf-8\\\\\\\')\\\\n    records = []\\\\n    for i, p in enumerate(tqdm(items, desc=\\\\\\\'缓存两种固定输出（可复用）\\\\\\\')):\\\\n        dest = folder / f\\\\\\\'{i:03d}.npz\\\\\\\'\\\\n        if not dest.is_file():\\\\n            x = J.tensor(p[\\\\\\\'x\\\\\\\'])\\\\n            c = torch.stack([torch.from_numpy(J.H.predict(model, p[\\\\\\\'x\\\\\\\'], s)).permute(2, 0, 1).cuda() for s in (0.5, 0.75, 1.0)])[None]\\\\n            lows = J.fusion_lowpass(c[0])[None]\\\\n            old = J.arr(band(x, c, lows))\\\\n            original = J.arr(c[:, 2])\\\\n            tmp = dest.with_suffix(\\\\\\\'.tmp\\\\\\\')\\\\n            with tmp.open(\\\\\\\'wb\\\\\\\') as f:\\\\n                np.savez(f, x=p[\\\\\\\'x\\\\\\\'], gt=p[\\\\\\\'gt\\\\\\\'], original=original.astype(np.float16), old_band=old.astype(np.float16))\\\\n            tmp.replace(dest)\\\\n        records.append({k: v for k, v in p.items() if k not in [\\\\\\\'x\\\\\\\', \\\\\\\'gt\\\\\\\']} | {\\\\\\\'cache\\\\\\\': str(dest)})\\\\n    return (records, signature)\\\\n\\\\ndef features(x, b):\\\\n    axes = (0, 1)\\\\n    return np.concatenate([x.mean(axes), x.std(axes), b.mean(axes), b.std(axes), (x - b).mean(axes), (x - b).std(axes)]).astype(np.float64)\\\\n\\\\ndef weighted_fit(X, Y, domains, alpha):\\\\n    groups = sorted(set(domains))\\\\n    w = np.array([1.0 / (len(groups) * sum((d == g for d in domains))) for g in domains])\\\\n    mean = (w[:, None] * X).sum(0)\\\\n    std = np.sqrt((w[:, None] * (X - mean) ** 2).sum(0))\\\\n    std = np.maximum(std, 0.01)\\\\n    A = np.column_stack([(X - mean) / std, np.ones(len(X))])\\\\n    penalty = np.eye(A.shape[1]) * alpha\\\\n    penalty[-1, -1] = 0.0\\\\n    at = torch.from_numpy(A)\\\\n    wt = torch.from_numpy(w[:, None])\\\\n    yt = torch.from_numpy(Y)\\\\n    coeff = torch.linalg.solve(at.T @ (wt * at) + torch.from_numpy(penalty) / len(X), at.T @ (wt * yt)).numpy()\\\\n    return {\\\\\\\'mean\\\\\\\': mean, \\\\\\\'std\\\\\\\': std, \\\\\\\'coeff\\\\\\\': coeff}\\\\n\\\\ndef infer_shift(f, fit):\\\\n    row = np.r_[(f - fit[\\\\\\\'mean\\\\\\\']) / fit[\\\\\\\'std\\\\\\\'], 1.0]\\\\n    return np.clip((row[:, None] * fit[\\\\\\\'coeff\\\\\\\']).sum(0), -MAX_SHIFT, MAX_SHIFT)\\\\n\\\\ndef evaluate(records, fits, domain_means, out):\\\\n    rows = []\\\\n    for p in tqdm([r for r in records if r[\\\\\\\'split\\\\\\\'] == \\\\\\\'val\\\\\\\'], desc=\\\\\\\'CPU评价\\\\\\\'):\\\\n        with np.load(p[\\\\\\\'cache\\\\\\\']) as a:\\\\n            x = a[\\\\\\\'x\\\\\\\'].astype(np.float32) / 255\\\\n            gt = a[\\\\\\\'gt\\\\\\\'].astype(np.float32) / 255\\\\n            predictions = {\\\\\\\'input\\\\\\\': x}\\\\n            for base in [\\\\\\\'original\\\\\\\', \\\\\\\'old_band\\\\\\\']:\\\\n                b = a[base].astype(np.float32)\\\\n                feat = features(x, b)\\\\n                predictions[base] = b\\\\n                predictions[base + \\\\\\\'_zero_shift\\\\\\\'] = np.clip(b + np.zeros(3, dtype=np.float32), 0, 1)\\\\n                predictions[base + \\\\\\\'_mean_anchor\\\\\\\'] = np.clip(b + (x - b).mean((0, 1), keepdims=True), 0, 1)\\\\n                predictions[base + \\\\\\\'_domain_constant\\\\\\\'] = np.clip(b + np.clip(domain_means[base, p[\\\\\\\'domain\\\\\\\']], -MAX_SHIFT, MAX_SHIFT), 0, 1)\\\\n                for alpha in RIDGE_ALPHAS:\\\\n                    predictions[f\\\\\\\'{base}_ridge_{alpha:g}\\\\\\\'] = np.clip(b + infer_shift(feat, fits[base, alpha]), 0, 1)\\\\n        for method, pred in predictions.items():\\\\n            rows.append({\\\\\\\'domain\\\\\\\': p[\\\\\\\'domain\\\\\\\'], \\\\\\\'id\\\\\\\': p[\\\\\\\'id\\\\\\\'], \\\\\\\'method\\\\\\\': method, **J.H.metrics(pred, gt)})\\\\n        i = sum((r[\\\\\\\'domain\\\\\\\'] == p[\\\\\\\'domain\\\\\\\'] and r[\\\\\\\'split\\\\\\\'] == \\\\\\\'val\\\\\\\' and (r[\\\\\\\'id\\\\\\\'] < p[\\\\\\\'id\\\\\\\']) for r in records))\\\\n        chosen = [\\\\\\\'input\\\\\\\', \\\\\\\'original\\\\\\\', \\\\\\\'original_mean_anchor\\\\\\\', f\\\\\\\'original_ridge_{PRIMARY_ALPHA:g}\\\\\\\', \\\\\\\'old_band\\\\\\\', f\\\\\\\'old_band_ridge_{PRIMARY_ALPHA:g}\\\\\\\']\\\\n        for method in chosen:\\\\n            dest = out / \\\\\\\'images\\\\\\\' / p[\\\\\\\'domain\\\\\\\'] / method\\\\n            dest.mkdir(parents=True, exist_ok=True)\\\\n            Image.fromarray(np.round(predictions[method] * 255).astype(\\\\\\\'uint8\\\\\\\')).save(dest / f\\\\\\\'{i:02d}.png\\\\\\\')\\\\n        dest = out / \\\\\\\'previews\\\\\\\'\\\\n        dest.mkdir(exist_ok=True)\\\\n        J.H.preview([(k, predictions[k]) for k in chosen] + [(\\\\\\\'GT\\\\\\\', gt)], dest / f\\\\\\\'{p[\\\\\\\'domain\\\\\\\']}_{i:02d}.jpg\\\\\\\')\\\\n        pd.DataFrame(rows).to_csv(out / \\\\\\\'per_image.csv\\\\\\\', index=False)\\\\n    frame = pd.DataFrame(rows)\\\\n    for base in [\\\\\\\'original\\\\\\\', \\\\\\\'old_band\\\\\\\']:\\\\n        a = frame[frame.method == base].set_index(\\\\\\\'id\\\\\\\')\\\\n        b = frame[frame.method == base + \\\\\\\'_zero_shift\\\\\\\'].set_index(\\\\\\\'id\\\\\\\')\\\\n        assert np.allclose(a.psnr, b.psnr, atol=1e-09), \\\\\\\'零修正对照不一致\\\\\\\'\\\\n    table = frame.groupby([\\\\\\\'domain\\\\\\\', \\\\\\\'method\\\\\\\']).agg(psnr=(\\\\\\\'psnr\\\\\\\', \\\\\\\'mean\\\\\\\'), ssim=(\\\\\\\'ssim\\\\\\\', \\\\\\\'mean\\\\\\\'), images=(\\\\\\\'id\\\\\\\', \\\\\\\'nunique\\\\\\\'))\\\\n    table.to_csv(out / \\\\\\\'summary.csv\\\\\\\')\\\\n    display(table)\\\\n    return frame\\\\n\\\\ndef run_calibration():\\\\n    assert PRIMARY_ALPHA in RIDGE_ALPHAS\\\\n    ROOT.mkdir(parents=True, exist_ok=True)\\\\n    out = ROOT / datetime.now().strftime(\\\\\\\'calibration_%Y%m%d_%H%M%S\\\\\\\')\\\\n    out.mkdir()\\\\n    try:\\\\n        items, model, band, meta = prepare()\\\\n        records, signature = cache_predictions(items, model, band, meta)\\\\n        model.cpu()\\\\n        band.cpu()\\\\n        del model, band, items\\\\n        torch.cuda.empty_cache()\\\\n        print(\\\\\\\'固定预测缓存完成；后续在CPU拟合及评价，不再反向传播主干。\\\\\\\')\\\\n        pd.DataFrame(records).to_csv(out / \\\\\\\'manifest.csv\\\\\\\', index=False)\\\\n        settings = {\\\\\\\'signature\\\\\\\': signature, \\\\\\\'primary_alpha\\\\\\\': PRIMARY_ALPHA, \\\\\\\'alphas\\\\\\\': RIDGE_ALPHAS, \\\\\\\'max_shift\\\\\\\': MAX_SHIFT, \\\\\\\'fit\\\\\\\': \\\\\\\'18 input/prediction statistics -> 3 global RGB shifts; ridge; equal domain weight; train only\\\\\\\', \\\\\\\'gt_label\\\\\\\': \\\\\\\'mean(GT - clipped_base) over native center768; preclipping global MSE optimum\\\\\\\', \\\\\\\'domain_constant\\\\\\\': \\\\\\\'separate train mean shift per known dataset; uses dataset label, only a comparator\\\\\\\', \\\\\\\'precision\\\\\\\': \\\\\\\'float16 frozen predictions cached then float32 evaluation; not pixel-identical to prior float32 baseline\\\\\\\', \\\\\\\'limits\\\\\\\': \\\\\\\'same 12/8 DCID image split and 20/10 UHDM development split as previous run; no independent test; scene overlap unknown; alpha10 primary, others development sensitivity; not guaranteed gain\\\\\\\'}\\\\n        (out / \\\\\\\'protocol.json\\\\\\\').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding=\\\\\\\'utf-8\\\\\\\')\\\\n        train = [p for p in records if p[\\\\\\\'split\\\\\\\'] == \\\\\\\'train\\\\\\\']\\\\n        fits = {}\\\\n        domain_means = {}\\\\n        feature_rows = []\\\\n        for base in [\\\\\\\'original\\\\\\\', \\\\\\\'old_band\\\\\\\']:\\\\n            X = []\\\\n            Y = []\\\\n            domains = []\\\\n            for p in train:\\\\n                with np.load(p[\\\\\\\'cache\\\\\\\']) as a:\\\\n                    x = a[\\\\\\\'x\\\\\\\'].astype(np.float64) / 255\\\\n                    g = a[\\\\\\\'gt\\\\\\\'].astype(np.float64) / 255\\\\n                    b = a[base].astype(np.float64)\\\\n                    f = features(x, b)\\\\n                    y = (g - b).mean((0, 1))\\\\n                X.append(f)\\\\n                Y.append(y)\\\\n                domains.append(p[\\\\\\\'domain\\\\\\\'])\\\\n                feature_rows.append({\\\\\\\'id\\\\\\\': p[\\\\\\\'id\\\\\\\'], \\\\\\\'domain\\\\\\\': p[\\\\\\\'domain\\\\\\\'], \\\\\\\'base\\\\\\\': base, **{f\\\\\\\'f{i}\\\\\\\': v for i, v in enumerate(f)}, **{f\\\\\\\'label{i}\\\\\\\': v for i, v in enumerate(y)}})\\\\n            X, Y = (np.array(X), np.array(Y))\\\\n            for d in sorted(set(domains)):\\\\n                domain_means[base, d] = Y[np.array(domains) == d].mean(0)\\\\n            for alpha in RIDGE_ALPHAS:\\\\n                fit = weighted_fit(X, Y, domains, alpha)\\\\n                fits[base, alpha] = fit\\\\n                folder = out / \\\\\\\'models\\\\\\\'\\\\n                folder.mkdir(exist_ok=True)\\\\n                np.savez(folder / f\\\\\\\'{base}_ridge_{alpha:g}.npz\\\\\\\', **fit, max_shift=MAX_SHIFT)\\\\n        pd.DataFrame(feature_rows).to_csv(out / \\\\\\\'train_statistics.csv\\\\\\\', index=False)\\\\n        (out / \\\\\\\'domain_constant.json\\\\\\\').write_text(json.dumps({f\\\\\\\'{b}/{d}\\\\\\\': v.tolist() for (b, d), v in domain_means.items()}, indent=2))\\\\n        evaluate(records, fits, domain_means, out)\\\\n        print(\\\\\\\'完成。先比较固定alpha10与各自base、mean_anchor；同时检查另一域有无退化，不自动挑验证集最高分。\\\\\\\')\\\\n    except Exception as error:\\\\n        (out / \\\\\\\'error.txt\\\\\\\').write_text(repr(error), encoding=\\\\\\\'utf-8\\\\\\\')\\\\n        raise\\\\n    finally:\\\\n        bundle = out.parent / (out.name + \\\\\\\'_results.zip\\\\\\\')\\\\n        with zipfile.ZipFile(bundle, \\\\\\\'w\\\\\\\', zipfile.ZIP_DEFLATED) as z:\\\\n            for p in out.rglob(\\\\\\\'*\\\\\\\'):\\\\n                if p.is_file():\\\\n                    z.write(p, p.relative_to(out))\\\\n        print(\\\\\\\'结果ZIP：\\\\\\\', bundle)\\\\n        display(FileLink(str(bundle)))\\\\n    return out\\\', C.__dict__)\\nfrom pathlib import Path\\nfrom datetime import datetime\\nimport json, random, hashlib, time, zipfile, copy\\nimport numpy as np\\nimport pandas as pd\\nfrom PIL import Image\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nfrom tqdm.auto import tqdm\\nfrom IPython.display import display, FileLink\\nCACHE_DIR = None\\nDCID_SOURCE = None\\nFUSION_ZIP = None\\nUHDM_ROOT = None\\nWEIGHT = None\\nSTEPS = 1000\\nEVAL_EVERY = 250\\nBATCH = 4\\nGRID = 96\\nLR = 0.0003\\nSEED = 2026\\nTV_WEIGHT = 0.001\\nMAX_RESIDUAL = 0.35\\nOUT_ROOT = Path(\\\'/kaggle/working/spatial_moire\\\')\\n\\ndef get_cache():\\n    if CACHE_DIR is not None:\\n        folders = [Path(CACHE_DIR)]\\n    else:\\n        roots = [Path(\\\'/kaggle/working/color_calibration/cache\\\')]\\n        folders = [p.parent for root in roots if root.exists() for p in root.glob(\\\'*/signature.json\\\')]\\n    if len(folders) > 1:\\n        raise RuntimeError(f\\\'缓存不唯一，请填写CACHE_DIR：{folders}\\\')\\n    if folders:\\n        folder = folders[0]\\n        sig = json.loads((folder / \\\'signature.json\\\').read_text())\\n        assert sig[\\\'protocol\\\'] == \\\'clipped_float16_original_and_band_v1\\\' and sig[\\\'region\\\'] == 768, \\\'缓存协议不匹配\\\'\\n        records = [dict(r, cache=str(folder / f\\\'{i:03d}.npz\\\')) for i, r in enumerate(sig[\\\'samples\\\'])]\\n        if not all((Path(r[\\\'cache\\\']).is_file() for r in records)):\\n            raise RuntimeError(\\\'缓存不完整；恢复缺失文件，或将该缓存目录另存后重新建立缓存\\\')\\n        print(\\\'复用上一轮缓存：\\\', folder)\\n        return (records, sig)\\n    C.DCID_SOURCE = DCID_SOURCE\\n    C.FUSION_ZIP = FUSION_ZIP\\n    C.UHDM_ROOT = UHDM_ROOT\\n    C.WEIGHT = WEIGHT\\n    items, model, band, meta = C.prepare()\\n    records, sig = C.cache_predictions(items, model, band, meta)\\n    model.cpu()\\n    band.cpu()\\n    del items, model, band\\n    torch.cuda.empty_cache()\\n    return (records, sig)\\n\\ndef load_memory(records):\\n    expected = {(\\\'DCID\\\', \\\'train\\\'): 12, (\\\'DCID\\\', \\\'val\\\'): 8, (\\\'UHDM\\\', \\\'train\\\'): 20, (\\\'UHDM\\\', \\\'val\\\'): 10}\\n    counts = {k: sum(((r[\\\'domain\\\'], r[\\\'split\\\']) == k for r in records)) for k in expected}\\n    assert counts == expected and len(records) == 50, f\\\'样本划分不匹配：{counts}\\\'\\n    C.J.check_split([r for r in records if r[\\\'split\\\'] == \\\'train\\\'], [r for r in records if r[\\\'split\\\'] == \\\'val\\\'])\\n    assert len({r[\\\'id\\\'] for r in records}) == 50, \\\'样本ID重复\\\'\\n    items = []\\n    for r in tqdm(records, desc=\\\'检查并载入缓存\\\'):\\n        with np.load(r[\\\'cache\\\']) as z:\\n            a = {k: np.array(z[k], copy=True) for k in [\\\'x\\\', \\\'gt\\\', \\\'original\\\', \\\'old_band\\\']}\\n        for k, v in a.items():\\n            assert v.shape == (768, 768, 3) and np.isfinite(v).all(), f\\\'{r[\\\'id\\\']}/{k} 异常\\\'\\n            if k in [\\\'x\\\', \\\'gt\\\']:\\n                assert v.dtype == np.uint8\\n            else:\\n                assert v.min() >= 0 and v.max() <= 1\\n        items.append(dict(r, arrays=a))\\n    return items\\n\\ndef tensors(p, flip=0):\\n    output = []\\n    for k in [\\\'x\\\', \\\'gt\\\', \\\'original\\\', \\\'old_band\\\']:\\n        a = p[\\\'arrays\\\'][k]\\n        if flip & 1:\\n            a = np.flip(a, 1)\\n        if flip & 2:\\n            a = np.flip(a, 0)\\n        t = torch.from_numpy(a.copy()).permute(2, 0, 1)[None].float().cuda()\\n        output.append(t / 255 if k in [\\\'x\\\', \\\'gt\\\'] else t)\\n    return output\\n\\ndef candidates(x, o, b):\\n\\n    def mean_anchor(y):\\n        return (y + (x - y).mean((-2, -1), keepdim=True)).clamp(0, 1)\\n    return torch.stack([o, b, mean_anchor(o), mean_anchor(b)], dim=1)\\n\\nclass LowGridNet(nn.Module):\\n\\n    def __init__(self, outputs):\\n        super().__init__()\\n        self.enc = nn.Sequential(nn.Conv2d(12, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, 24, 3, padding=1), nn.ReLU())\\n        self.down = nn.Sequential(nn.Conv2d(24, 48, 3, stride=2, padding=1), nn.ReLU(), nn.Conv2d(48, 48, 3, padding=1), nn.ReLU())\\n        self.up = nn.Sequential(nn.Conv2d(72, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, 24, 3, padding=1), nn.ReLU())\\n        self.final = nn.Conv2d(24, outputs, 1)\\n        nn.init.zeros_(self.final.weight)\\n        nn.init.zeros_(self.final.bias)\\n\\n    def forward(self, f):\\n        e = self.enc(f)\\n        d = self.down(e)\\n        return self.final(self.up(torch.cat([e, F.interpolate(d, size=e.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)], dim=1)))\\n\\nclass RepairHead(nn.Module):\\n\\n    def __init__(self, mode):\\n        super().__init__()\\n        self.mode = mode\\n        self.net = LowGridNet(3 if mode == \\\'spatial\\\' else 4)\\n        if mode == \\\'gate\\\':\\n            with torch.no_grad():\\n                self.net.final.bias[1] = 2.0\\n\\n    def forward(self, x, o, b):\\n        features = torch.cat([x, o, b, x - b], dim=1)\\n        f = F.interpolate(features, size=(GRID, GRID), mode=\\\'area\\\')\\n        small = self.net(f)\\n        if self.mode == \\\'spatial\\\':\\n            field = MAX_RESIDUAL * torch.tanh(small)\\n            y = b + F.interpolate(field, size=b.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n        else:\\n            field = small.softmax(1)\\n            weights = F.interpolate(field, size=b.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n            y = (weights.unsqueeze(2) * candidates(x, o, b)).sum(1)\\n        return (y, field)\\n\\ndef tv(field):\\n    return (field[:, :, 1:] - field[:, :, :-1]).abs().mean() + (field[:, :, :, 1:] - field[:, :, :, :-1]).abs().mean()\\n\\ndef pack(out):\\n    bundle = out.parent / (out.name + \\\'_results.zip\\\')\\n    with zipfile.ZipFile(bundle, \\\'w\\\', zipfile.ZIP_DEFLATED) as z:\\n        for p in out.rglob(\\\'*\\\'):\\n            if p.is_file():\\n                z.write(p, p.relative_to(out))\\n    print(\\\'下载结果包：\\\', bundle)\\n    display(FileLink(str(bundle)))\\n\\n@torch.inference_mode()\\ndef evaluate(items, out, method, step, net=None, ssim=True):\\n    if net is not None:\\n        net.eval()\\n    rows = []\\n    for i, p in enumerate([p for p in items if p[\\\'split\\\'] == \\\'val\\\']):\\n        x, g, o, b = tensors(p)\\n        if net is None:\\n            c = candidates(x, o, b)\\n            preds = {\\\'input\\\': x, \\\'original\\\': c[:, 0], \\\'old_band\\\': c[:, 1], \\\'original_mean_anchor\\\': c[:, 2], \\\'old_band_mean_anchor\\\': c[:, 3]}\\n        else:\\n            preds = {method: net(x, o, b)[0].clamp(0, 1)}\\n        gt = C.J.arr(g)\\n        for label, pred in preds.items():\\n            image = C.J.arr(pred)\\n            if ssim:\\n                metric = C.J.H.metrics(image, gt)\\n            else:\\n                metric = {\\\'psnr\\\': float(-10 * np.log10(max(np.mean((image.astype(np.float64) - gt.astype(np.float64)) ** 2), 1e-12))), \\\'ssim\\\': float(\\\'nan\\\')}\\n            rows.append({\\\'domain\\\': p[\\\'domain\\\'], \\\'id\\\': p[\\\'id\\\'], \\\'method\\\': label, \\\'step\\\': step, **metric})\\n            if ssim:\\n                dest = out / \\\'images\\\' / p[\\\'domain\\\'] / label / f\\\'step_{step}\\\'\\n                dest.mkdir(parents=True, exist_ok=True)\\n                Image.fromarray(np.round(image * 255).astype(\\\'uint8\\\')).save(dest / f\\\'{i:02d}.png\\\')\\n        if ssim:\\n            views = [(\\\'Input\\\', C.J.arr(x)), (\\\'Old band\\\', C.J.arr(b))]\\n            views += [(k, C.J.arr(v)) for k, v in preds.items() if k not in [\\\'input\\\', \\\'old_band\\\']]\\n            dest = out / \\\'previews\\\'\\n            dest.mkdir(exist_ok=True)\\n            C.J.H.preview(views + [(\\\'GT\\\', gt)], dest / f\\\'{p[\\\'domain\\\']}_{method}_{step}_{i:02d}.jpg\\\')\\n    return rows\\n\\ndef persist(out, rows, history):\\n    frame = pd.DataFrame(rows)\\n    frame.to_csv(out / \\\'per_image.csv\\\', index=False)\\n    summary = frame.groupby([\\\'domain\\\', \\\'method\\\', \\\'step\\\']).agg(psnr=(\\\'psnr\\\', \\\'mean\\\'), ssim=(\\\'ssim\\\', \\\'mean\\\'), images=(\\\'id\\\', \\\'nunique\\\'), ssim_images=(\\\'ssim\\\', \\\'count\\\'))\\n    summary.to_csv(out / \\\'summary.csv\\\')\\n    pd.DataFrame(history).to_csv(out / \\\'history.csv\\\', index=False)\\n    return summary\\n\\ndef run_spatial():\\n    assert torch.cuda.is_available(), \\\'需要GPU训练小分支\\\'\\n    assert BATCH > 0 and BATCH % 2 == 0 and (GRID >= 4)\\n    OUT_ROOT.mkdir(parents=True, exist_ok=True)\\n    out = OUT_ROOT / datetime.now().strftime(\\\'spatial_%Y%m%d_%H%M%S\\\')\\n    out.mkdir()\\n    rows = []\\n    history = []\\n    try:\\n        records, sig = get_cache()\\n        items = load_memory(records)\\n        pd.DataFrame(records).to_csv(out / \\\'manifest.csv\\\', index=False)\\n        config = {\\\'seed\\\': SEED, \\\'steps_per_head\\\': STEPS, \\\'batch\\\': BATCH, \\\'grid\\\': GRID, \\\'lr\\\': LR, \\\'tv_weight\\\': TV_WEIGHT, \\\'max_residual\\\': MAX_RESIDUAL, \\\'signature\\\': sig, \\\'region\\\': 768, \\\'input\\\': \\\'center768 only; downsample whole center view to96; NOT original full-image guidance\\\', \\\'sampling\\\': \\\'equal batch counts DCID/UHDM; full center regions; synchronized horizontal/vertical flips\\\', \\\'base\\\': \\\'frozen old_band for spatial; gate convex weights over original,old_band,and their input-mean corrected outputs\\\', \\\'loss\\\': \\\'unclipped RGB MSE + TV on low-resolution residual field or gate weights; clipped output for metrics\\\', \\\'selection\\\': \\\'fixed last step primary; best macro development improvement saved separately\\\', \\\'limits\\\': \\\'12/8 target and20/10 source development splits; prior inspected; no independent scene-generalization claim\\\', \\\'torch\\\': torch.__version__, \\\'gpu\\\': torch.cuda.get_device_name(0)}\\n        (out / \\\'protocol.json\\\').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\n        print(\\\'阶段1：强基线（含SSIM）\\\')\\n        rows = evaluate(items, out, \\\'baseline\\\', 0)\\n        baseline = pd.DataFrame(rows).groupby([\\\'domain\\\', \\\'method\\\']).psnr.mean()\\n        references = {\\\'DCID\\\': float(baseline[\\\'DCID\\\', \\\'original_mean_anchor\\\']), \\\'UHDM\\\': float(baseline[\\\'UHDM\\\', \\\'old_band\\\'])}\\n        display(persist(out, rows, history))\\n        pack(out)\\n        pools = {d: [p for p in items if p[\\\'split\\\'] == \\\'train\\\' and p[\\\'domain\\\'] == d] for d in [\\\'DCID\\\', \\\'UHDM\\\']}\\n        for mode in [\\\'gate\\\', \\\'spatial\\\']:\\n            random.seed(SEED)\\n            np.random.seed(SEED)\\n            torch.manual_seed(SEED)\\n            torch.cuda.manual_seed_all(SEED)\\n            net = RepairHead(mode).cuda()\\n            optimizer = torch.optim.Adam(net.parameters(), lr=LR)\\n            rng = random.Random(SEED)\\n            parameter_count = sum((p.numel() for p in net.parameters()))\\n            print(mode, \\\'参数量\\\', parameter_count)\\n            x, g, o, b = tensors(pools[\\\'DCID\\\'][0])\\n            initial, field = net(x, o, b)\\n            if mode == \\\'spatial\\\':\\n                assert torch.equal(initial, b), \\\'空间分支零初始化未复现基础输出\\\'\\n            del x, g, o, b, initial, field\\n            initial_rows = evaluate(items, out, mode, 0, net=net, ssim=True)\\n            rows += initial_rows\\n            initial_score = np.mean([np.mean([r[\\\'psnr\\\'] for r in initial_rows if r[\\\'domain\\\'] == d]) - references[d] for d in references])\\n            best = float(initial_score)\\n            payload = lambda step: {\\\'model\\\': net.state_dict(), \\\'mode\\\': mode, \\\'step\\\': step, \\\'config\\\': config, \\\'parameters\\\': parameter_count}\\n            torch.save(payload(0), out / (mode + \\\'_best_dev.pt\\\'))\\n            persist(out, rows, history)\\n            start = time.perf_counter()\\n            recent = []\\n            for step in tqdm(range(1, STEPS + 1), desc=mode + \\\'训练\\\'):\\n                selected = [rng.choice(pools[d]) for d in [\\\'DCID\\\', \\\'UHDM\\\'] for _ in range(BATCH // 2)]\\n                samples = [tensors(p, rng.randrange(4)) for p in selected]\\n                x, g, o, b = [torch.cat([s[j] for s in samples]) for j in range(4)]\\n                net.train()\\n                optimizer.zero_grad(set_to_none=True)\\n                pred, field = net(x, o, b)\\n                mse = F.mse_loss(pred, g)\\n                smooth = tv(field)\\n                loss = mse + TV_WEIGHT * smooth\\n                if not torch.isfinite(loss):\\n                    raise RuntimeError(\\\'训练loss非有限\\\')\\n                loss.backward()\\n                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)\\n                optimizer.step()\\n                recent.append(float(mse.detach()))\\n                del samples, pred, field, x, g, o, b, loss, mse, smooth\\n                if step == 20:\\n                    torch.cuda.synchronize()\\n                    elapsed = time.perf_counter() - start\\n                    print(f\\\'{mode}前20步{elapsed:.1f}s，剩余训练约{(STEPS - step) * elapsed / 20 / 60:.1f}分钟，另有评价时间\\\')\\n                if step % EVAL_EVERY == 0 or step == STEPS:\\n                    scored = evaluate(items, out, mode, step, net, ssim=step == STEPS)\\n                    rows += scored\\n                    means = {d: float(np.mean([r[\\\'psnr\\\'] for r in scored if r[\\\'domain\\\'] == d])) for d in references}\\n                    score = float(np.mean([means[d] - references[d] for d in references]))\\n                    history.append({\\\'mode\\\': mode, \\\'step\\\': step, \\\'train_mse_recent\\\': float(np.mean(recent)), \\\'DCID_psnr\\\': means[\\\'DCID\\\'], \\\'UHDM_psnr\\\': means[\\\'UHDM\\\'], \\\'macro_gain_vs_strong_refs\\\': score, \\\'elapsed_seconds\\\': time.perf_counter() - start})\\n                    recent = []\\n                    torch.save(payload(step), out / (mode + \\\'_last.pt\\\'))\\n                    if score > best:\\n                        best = score\\n                        torch.save(payload(step), out / (mode + \\\'_best_dev.pt\\\'))\\n                    print(mode, step, means)\\n                    persist(out, rows, history)\\n            del optimizer, net\\n            torch.cuda.empty_cache()\\n            pack(out)\\n        display(persist(out, rows, history))\\n        print(\\\'完成。固定1000步是主结果；中间步骤仅PSNR，NaN SSIM表示未计算。\\\')\\n    except Exception as error:\\n        (out / \\\'error.txt\\\').write_text(repr(error), encoding=\\\'utf-8\\\')\\n        if rows:\\n            persist(out, rows, history)\\n        raise\\n    finally:\\n        pack(out)\\n    return out\', S.__dict__)\nfrom pathlib import Path\nfrom datetime import datetime\nimport json, random, time\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom tqdm.auto import tqdm\nfrom PIL import Image\nfrom IPython.display import display\nCACHE_DIR = None\nDCID_SOURCE = None\nFUSION_ZIP = None\nUHDM_ROOT = None\nWEIGHT = None\nSTEPS = 500\nEVAL_EVERY = 100\nBATCH = 4\nGRID = 96\nLR = 0.0003\nTV_WEIGHT = 0.001\nMAX_RESIDUAL = 0.35\nSEED = 2026\nROOT = Path(\'/kaggle/working/independent_repair\')\n\ndef base_for_domain(domain, x, o, b):\n    if domain == \'DCID\':\n        return (o + (x - o).mean((-2, -1), keepdim=True)).clamp(0, 1)\n    if domain == \'UHDM\':\n        return b\n    raise ValueError(domain)\n\nclass IndependentRepair(nn.Module):\n\n    def __init__(self, domain):\n        super().__init__()\n        self.domain = domain\n        self.net = S.LowGridNet(3)\n\n    def forward(self, x, o, b):\n        base = base_for_domain(self.domain, x, o, b)\n        f = F.interpolate(torch.cat([x, o, base, x - base], 1), size=(GRID, GRID), mode=\'area\')\n        field = MAX_RESIDUAL * torch.tanh(self.net(f))\n        return (base + F.interpolate(field, size=base.shape[-2:], mode=\'bilinear\', align_corners=False), field)\n\ndef persist(out, rows, history):\n    frame = pd.DataFrame(rows)\n    frame.to_csv(out / \'per_image.csv\', index=False)\n    table = frame.groupby([\'domain\', \'split\', \'method\', \'step\']).agg(psnr=(\'psnr\', \'mean\'), ssim=(\'ssim\', \'mean\'), images=(\'id\', \'nunique\'), ssim_images=(\'ssim\', \'count\'), raw_mse=(\'raw_mse\', \'mean\'), objective=(\'objective\', \'mean\'), field_tv=(\'field_tv\', \'mean\'))\n    table.to_csv(out / \'summary.csv\')\n    pd.DataFrame(history).to_csv(out / \'history.csv\', index=False)\n    return table\n\n@torch.inference_mode()\ndef evaluate(items, domain, out, step, net=None):\n    if net is not None:\n        net.eval()\n    rows = []\n    for index, p in enumerate([p for p in items if p[\'domain\'] == domain]):\n        x, g, o, b = S.tensors(p)\n        base = base_for_domain(domain, x, o, b)\n        if net is None:\n            raw, field = (base, None)\n        else:\n            raw, field = net(x, o, b)\n        image = S.C.J.arr(raw)\n        gt = S.C.J.arr(g)\n        do_ssim = p[\'split\'] == \'val\' and step in (0, STEPS)\n        if do_ssim:\n            metric = S.C.J.H.metrics(image, gt)\n        else:\n            metric = {\'psnr\': float(-10 * np.log10(max(np.mean((image.astype(np.float64) - gt.astype(np.float64)) ** 2), 1e-12))), \'ssim\': float(\'nan\')}\n        raw_mse = float(F.mse_loss(raw, g))\n        smooth = 0.0 if field is None else float(S.tv(field))\n        method = \'strong_base\' if net is None else \'independent_spatial\'\n        rows.append({\'domain\': domain, \'split\': p[\'split\'], \'id\': p[\'id\'], \'method\': method, \'step\': step, **metric, \'raw_mse\': raw_mse, \'field_tv\': smooth, \'objective\': raw_mse + TV_WEIGHT * smooth})\n        if do_ssim:\n            dest = out / \'images\' / domain / method / f\'step_{step}\'\n            dest.mkdir(parents=True, exist_ok=True)\n            for name, a in [(\'input\', S.C.J.arr(x)), (\'gt\', gt), (\'base\', S.C.J.arr(base)), (\'output\', image)]:\n                Image.fromarray(np.round(np.clip(a, 0, 1) * 255).astype(\'uint8\')).save(dest / f\'{index:02d}_{name}.png\')\n            folder = out / \'previews\'\n            folder.mkdir(exist_ok=True)\n            S.C.J.H.preview([(\'Input\', S.C.J.arr(x)), (\'Strong base\', S.C.J.arr(base)), (\'Repair\', image), (\'GT\', gt)], folder / f\'{domain}_{method}_{step}_{index:02d}.jpg\')\n    return rows\n\ndef run_independent():\n    assert torch.cuda.is_available(), \'需要GPU训练；无需重训主干\'\n    assert BATCH > 0 and STEPS > 0 and (GRID >= 4)\n    for name in [\'CACHE_DIR\', \'DCID_SOURCE\', \'FUSION_ZIP\', \'UHDM_ROOT\', \'WEIGHT\']:\n        setattr(S, name, globals()[name])\n    ROOT.mkdir(parents=True, exist_ok=True)\n    out = ROOT / datetime.now().strftime(\'independent_%Y%m%d_%H%M%S\')\n    out.mkdir()\n    rows = []\n    history = []\n    try:\n        records, signature = S.get_cache()\n        items = S.load_memory(records)\n        pd.DataFrame(records).to_csv(out / \'manifest.csv\', index=False)\n        config = {\'steps_per_domain\': STEPS, \'eval_every\': EVAL_EVERY, \'batch\': BATCH, \'grid\': GRID, \'lr\': LR, \'tv_weight\': TV_WEIGHT, \'max_residual\': MAX_RESIDUAL, \'seed\': SEED, \'signature\': signature, \'base\': {\'DCID\': \'original_mean_anchor\', \'UHDM\': \'old_band\'}, \'train\': \'independent head per known source; full center768, flips; frozen cached backbones\', \'evaluation\': \'all fixed training images and all fixed validation images, without augmentation; raw MSE/objective and clipped PSNR; val SSIM at0/500\', \'selection\': \'fixed500 primary; validation-best including zero checkpoint saved as development selection only\', \'limits\': \'known-source adaptation, NOT blind domain identification; 12/8 DCID and20/10 UHDM prior-inspected image splits, unknown scene overlap, no independent test\', \'torch\': str(torch.__version__), \'gpu\': torch.cuda.get_device_name(0)}\n        (out / \'protocol.json\').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        for domain in [\'DCID\', \'UHDM\']:\n            print(\'开始独立适配：\', domain)\n            random.seed(SEED)\n            np.random.seed(SEED)\n            torch.manual_seed(SEED)\n            torch.cuda.manual_seed_all(SEED)\n            net = IndependentRepair(domain).cuda()\n            optimizer = torch.optim.Adam(net.parameters(), lr=LR)\n            pool = [p for p in items if p[\'domain\'] == domain and p[\'split\'] == \'train\']\n            x, g, o, b = S.tensors(pool[0])\n            initial, _ = net(x, o, b)\n            assert torch.equal(initial, base_for_domain(domain, x, o, b)), \'零初始化没有复现强基线\'\n            del x, g, o, b, initial\n            base_rows = evaluate(items, domain, out, 0)\n            zero_rows = [dict(r, method=\'independent_spatial\') for r in base_rows]\n            rows += base_rows + zero_rows\n            val_base = float(np.mean([r[\'psnr\'] for r in base_rows if r[\'split\'] == \'val\']))\n            best = val_base\n            parameter_count = sum((p.numel() for p in net.parameters()))\n            payload = lambda step: {\'model\': net.state_dict(), \'domain\': domain, \'step\': step, \'config\': config, \'parameters\': parameter_count}\n            torch.save(payload(0), out / (domain + \'_best_dev.pt\'))\n            print(\'参数量\', parameter_count, \'零初始化验证通过，验证基线\', val_base)\n            display(persist(out, rows, history))\n            S.pack(out)\n            rng = random.Random(SEED)\n            start = time.perf_counter()\n            recent = []\n            for step in tqdm(range(1, STEPS + 1), desc=domain + \'独立训练\'):\n                samples = [S.tensors(rng.choice(pool), rng.randrange(4)) for _ in range(BATCH)]\n                x, g, o, b = [torch.cat([s[j] for s in samples]) for j in range(4)]\n                net.train()\n                optimizer.zero_grad(set_to_none=True)\n                pred, field = net(x, o, b)\n                mse = F.mse_loss(pred, g)\n                loss = mse + TV_WEIGHT * S.tv(field)\n                if not torch.isfinite(loss):\n                    raise RuntimeError(\'loss非有限值\')\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)\n                optimizer.step()\n                recent.append(float(mse.detach()))\n                del samples, x, g, o, b, pred, field, mse, loss\n                if step == 20:\n                    torch.cuda.synchronize()\n                    elapsed = time.perf_counter() - start\n                    print(f\'前20步{elapsed:.1f}s，余下训练约{(STEPS - step) * elapsed / 20 / 60:.1f}分钟，另有评价时间\')\n                if step % EVAL_EVERY == 0 or step == STEPS:\n                    scored = evaluate(items, domain, out, step, net)\n                    rows += scored\n                    fixed = {s: {k: float(np.mean([r[k] for r in scored if r[\'split\'] == s])) for k in [\'psnr\', \'raw_mse\', \'objective\']} for s in [\'train\', \'val\']}\n                    val_score = fixed[\'val\'][\'psnr\']\n                    history.append({\'domain\': domain, \'step\': step, \'batch_mse_recent\': float(np.mean(recent)), \'fixed_train_psnr\': fixed[\'train\'][\'psnr\'], \'fixed_val_psnr\': val_score, \'fixed_train_raw_mse\': fixed[\'train\'][\'raw_mse\'], \'fixed_val_raw_mse\': fixed[\'val\'][\'raw_mse\'], \'fixed_train_objective\': fixed[\'train\'][\'objective\'], \'fixed_val_objective\': fixed[\'val\'][\'objective\'], \'val_gain_vs_strong_base\': val_score - val_base, \'elapsed_seconds\': time.perf_counter() - start})\n                    recent = []\n                    torch.save(payload(step), out / (domain + \'_last.pt\'))\n                    if val_score > best:\n                        best = val_score\n                        torch.save(payload(step), out / (domain + \'_best_dev.pt\'))\n                    print(domain, step, \'固定训练/验证PSNR\', fixed[\'train\'][\'psnr\'], val_score, \'相对强基线\', val_score - val_base)\n                    persist(out, rows, history)\n            del optimizer, net\n            torch.cuda.empty_cache()\n            S.pack(out)\n        display(persist(out, rows, history))\n        print(\'完成。主结果固定500步；同时检查固定训练集与验证集的差异。最佳开发检查点可能仍为step0，代码不会强行选择退化模型。\')\n    except Exception as error:\n        (out / \'error.txt\').write_text(repr(error), encoding=\'utf-8\')\n        if rows:\n            persist(out, rows, history)\n        raise\n    finally:\n        S.pack(out)\n    return out',I.__dict__)
from pathlib import Path
from datetime import datetime
import io,json,random,hashlib,zipfile
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm
from huggingface_hub import HfApi,hf_hub_download
from IPython.display import display,FileLink

# 固定抽样与模型，不根据本轮成绩更改。
FROZEN_BUNDLE=None   # ZIP或展开后含frozen_protocol.json的目录
UHDM_ROOT=None
WEIGHT=None
HF_REPO='Mrduckk/DCID'
HF_REVISION='51c674aa3f71c4cbbddb5cfcba9fbe3536eb0722'
HF_SUBDIR='test/iphone_1'
N_UHDM=20
N_DCID=20
SEED=20261108
DOWNLOAD_LIMIT_MIB=200
ROOT=Path('/kaggle/working/frozen_moire_test')

def load_bundle():
    if FROZEN_BUNDLE is not None:p=Path(FROZEN_BUNDLE)
    else:
        roots=[Path('/kaggle/input'),Path('/kaggle/working')]
        candidates=[p for r in roots if r.exists() for p in r.rglob('moire_frozen_test_models.zip')]
        if not candidates:
            candidates=[p.parent for r in roots if r.exists() for p in r.rglob('frozen_protocol.json')]
        if len(candidates)!=1:raise RuntimeError(f'请挂载冻结测试权重包，或明确填写FROZEN_BUNDLE。候选={candidates}')
        p=candidates[0]
    if p.is_dir():read=lambda n:(p/n).read_bytes()
    else:
        z=zipfile.ZipFile(p);read=z.read
    frozen=json.loads(read('frozen_protocol.json'))
    protocol=json.loads(read('old_fusion/protocol.json'))
    band_data=read('old_fusion/band.pt');target_data=read('DCID_100.pt')
    assert hashlib.sha256(band_data).hexdigest()==frozen['band_sha256']
    assert hashlib.sha256(target_data).hexdigest()==frozen['DCID_100_sha256']
    band=torch.load(io.BytesIO(band_data),map_location='cpu',weights_only=True)
    target=torch.load(io.BytesIO(target_data),map_location='cpu',weights_only=True)
    assert target['domain']=='DCID' and target['step']==100
    return frozen,protocol,band,target

def choose_pairs(files,n,seed):
    pairs=[{'input':p[:-7]+'_m.jpg','target':p} for p in sorted(files) if p.endswith('_gt.jpg') and p[:-7]+'_m.jpg' in files]
    random.Random(seed).shuffle(pairs)
    if len(pairs)<n:raise RuntimeError(f'test目录仅找到{len(pairs)}对，需要{n}对；停止，不回退到train')
    return pairs[:n]

def download_target(out):
    assert HF_SUBDIR.startswith('test/'),'只能使用test池'
    api=HfApi()
    try:
        files={p.path:int(p.size) for p in api.list_repo_tree(HF_REPO,path_in_repo=HF_SUBDIR,
            repo_type='dataset',revision=HF_REVISION,recursive=True) if getattr(p,'size',None) is not None}
    except Exception as error:
        raise RuntimeError(f'无法读取锁定版本的{HF_SUBDIR}。检查Internet或HF访问；不会回退train。原错误：{error}') from error
    selected=choose_pairs(files,N_DCID,SEED)
    size=sum(files[p[k]] for p in selected for k in ['input','target'])
    plan={'repo':HF_REPO,'revision':HF_REVISION,'subdir':HF_SUBDIR,'seed':SEED,'bytes':size,'pairs':selected}
    (out/'DCID_download_plan.json').write_text(json.dumps(plan,indent=2),encoding='utf-8')
    print(f'DCID只下载{len(selected)}对，文件大小合计{size/1024**2:.1f}MiB')
    if size>DOWNLOAD_LIMIT_MIB*1024**2:raise RuntimeError('超过下载上限，未下载图片。不要根据结果更换抽样。')
    records=[]
    for pair in tqdm(selected,desc='按需获取DCID test'):
        paths={k:hf_hub_download(HF_REPO,filename=pair[k],repo_type='dataset',revision=HF_REVISION,
            local_dir=str(ROOT/'target_images')) for k in ['input','target']}
        records.append({'domain':'DCID','id':'DCID/'+pair['target'][:-7],**paths})
    return records

def source_pairs(root):
    gts=sorted(p for p in root.rglob('*gt.jpg') if any(part.lower() in ['test','testing','testset','test_data'] for part in p.relative_to(root).parts[:-1]))
    rows=[]
    for gt in gts:
        x=gt.with_name(gt.name[:4]+'_moire.jpg')
        if not x.is_file():raise RuntimeError(f'UHDM test缺少输入：{x}')
        rows.append({'domain':'UHDM','id':'UHDM/'+gt.relative_to(root).as_posix(),
                     'input':str(x),'target':str(gt)})
    if len(rows)<N_UHDM:raise RuntimeError(f'UHDM test仅找到{len(rows)}对；请挂载完整含test的UHDM，不回退train')
    random.Random(SEED).shuffle(rows)
    return rows[:N_UHDM]

def audit(records,root,old_protocol,target_ck,out):
    # 所有原融合训练100张及开发10张均加入精确重复检查，不只检查回放20张。
    prior={k:set() for k in ['input_hash','gt_hash']}
    for row in tqdm(old_protocol['manifest']['train']+old_protocol['manifest']['val'],desc='排查旧UHDM精确重复'):
        prior['input_hash'].add(I.S.C.J.H.sha_file(I.S.C.J.source_path(row['x'],root)))
        prior['gt_hash'].add(I.S.C.J.H.sha_file(I.S.C.J.source_path(row['gt'],root)))
    for row in target_ck['config']['signature']['samples']:
        prior['input_hash'].add(row['input_hash']);prior['gt_hash'].add(row['gt_hash'])
    seen={k:set() for k in prior};rows=[];issues=[]
    for p in tqdm(records,desc='测试配对和哈希检查'):
        r=dict(p)
        for key,hkey in [('input','input_hash'),('target','gt_hash')]:
            h=I.S.C.J.H.sha_file(p[key]);r[hkey]=h
            if h in prior[hkey]:issues.append({'id':p['id'],'issue':'exact_duplicate_of_prior_data','kind':key})
            if h in seen[hkey]:issues.append({'id':p['id'],'issue':'exact_duplicate_within_test','kind':key})
            seen[hkey].add(h)
        with Image.open(p['input']) as a,Image.open(p['target']) as b:
            a.load();b.load();r['width'],r['height']=a.size
            if a.size!=b.size or min(a.size)<768:issues.append({'id':p['id'],'issue':'invalid_size'})
        if r['input_hash']==r['gt_hash']:issues.append({'id':p['id'],'issue':'identical_input_target'})
        rows.append(r)
    pd.DataFrame(rows).to_csv(out/'test_manifest.csv',index=False)
    (out/'audit.json').write_text(json.dumps({'issues':issues,'pairs':len(rows),
        'limits':'hashes rule out exact files only, not same content across views. Author test split; scene independence not established.'},indent=2),encoding='utf-8')
    if issues:raise RuntimeError('配对/重复检查未通过；停止，不自动删图重抽，见audit.json')
    return rows

@torch.inference_mode()
def infer_all(records,model,band,repair,out):
    rows=[]
    for index,p in enumerate(tqdm(records,desc='冻结方法测试，不训练')):
        with Image.open(p['input']) as im:a=np.array(im.convert('RGB'))
        with Image.open(p['target']) as im:g=np.array(im.convert('RGB'))
        a,g=I.S.C.J.crop_pair(a,g,p['id']);x=I.S.C.J.tensor(a)
        candidates=torch.stack([torch.from_numpy(I.S.C.J.H.predict(model,a,s)).permute(2,0,1).cuda() for s in (.5,.75,1.)])[None]
        lows=I.S.C.J.fusion_lowpass(candidates[0])[None]
        b=band(x,candidates,lows).clamp(0,1);o=candidates[:,2]
        # 与开发缓存协议一致：固定输出float16量化后回float32，所有方法共用。
        o=o.half().float();b=b.half().float()
        anchored=(o+(x-o).mean((-2,-1),keepdim=True)).clamp(0,1)
        preds={'input':x,'original':o,'old_band':b,'original_mean_anchor':anchored}
        if p['domain']=='DCID':preds['DCID_spatial100']=repair(x,o,b)[0].clamp(0,1)
        gt=g.astype(np.float32)/255
        for label,value in preds.items():
            image=I.S.C.J.arr(value)
            rows.append({'domain':p['domain'],'id':p['id'],'method':label,**I.S.C.J.H.metrics(image,gt)})
            folder=out/'images'/p['domain']/f'{index:02d}';folder.mkdir(parents=True,exist_ok=True)
            Image.fromarray(np.round(image*255).astype('uint8')).save(folder/(label+'.png'))
        Image.fromarray(g).save(folder/'gt.png')
        folder=out/'previews';folder.mkdir(exist_ok=True)
        I.S.C.J.H.preview([(k,I.S.C.J.arr(v)) for k,v in preds.items()]+[('GT',gt)],folder/f'{p["domain"]}_{index:02d}.jpg')
        pd.DataFrame(rows).to_csv(out/'per_image.csv',index=False)
    frame=pd.DataFrame(rows)
    summary=frame.groupby(['domain','method']).agg(psnr=('psnr','mean'),ssim=('ssim','mean'),images=('id','nunique'))
    summary.to_csv(out/'summary.csv');display(summary)
    differences=[]
    for domain in ['DCID','UHDM']:
        pivot=frame[frame.domain==domain].pivot(index='id',columns='method',values='psnr')
        comparisons=[('old_band','original'),('original_mean_anchor','original')]
        if domain=='DCID':comparisons.append(('DCID_spatial100','original_mean_anchor'))
        for method,ref in comparisons:
            delta=pivot[method]-pivot[ref]
            differences.append({'domain':domain,'method':method,'reference':ref,'mean_gain':float(delta.mean()),
                'median_gain':float(delta.median()),'wins':int((delta>0).sum()),'images':len(delta)})
    pd.DataFrame(differences).to_csv(out/'paired_comparisons.csv',index=False)
    display(pd.DataFrame(differences))

def run_frozen_test():
    assert torch.cuda.is_available(),'需要GPU做固定推理，没有训练'
    ROOT.mkdir(parents=True,exist_ok=True)
    out=ROOT/datetime.now().strftime('frozen_%Y%m%d_%H%M%S');out.mkdir()
    try:
        frozen,old_protocol,band_ck,target_ck=load_bundle()
        if UHDM_ROOT is None:
            found=sorted(p for p in Path('/kaggle/input').rglob('uhdm-dataset') if p.is_dir())
            if len(found)!=1:raise RuntimeError('挂载soumikrakshit/uhdm-dataset，或设置UHDM_ROOT')
            source_root=found[0]
        else:source_root=Path(UHDM_ROOT)
        records=source_pairs(source_root)
        # 在看测试标签或成绩之前记录固定方法、权重指纹、区域和抽样协议。
        protocol={'frozen_models':frozen,'seed':SEED,'counts':{'UHDM':N_UHDM,'DCID':N_DCID},
            'HF_revision':HF_REVISION,'HF_subdir':HF_SUBDIR,'region':768,'train':False,
            'primary_methods':{'UHDM':'old_band vs original','DCID':'original_mean_anchor vs original'},
            'secondary':'DCID development-selected fixed100 spatial vs mean anchor; no test-based model selection',
            'precision':'same-view center768; fixed predictions float16 -> float32 as development cache',
            'limitations':'sampled author test subsets, not official full-image score; exact hashes do not prove scene/content independence'}
        (out/'protocol.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
        pd.DataFrame(records).to_csv(out/'UHDM_sampling_plan.csv',index=False)
        records+=download_target(out)
        records=audit(records,source_root,old_protocol,target_ck,out)
        H=I.S.C.J.H;H.ROOT=ROOT/'official';H.ROOT.mkdir(exist_ok=True);H.LOCAL_WEIGHT=WEIGHT
        if WEIGHT is None:
            known=[Path('/kaggle/working/independent_repair/official/uhdm_checkpoint.pth'),
                Path('/kaggle/working/color_calibration/official/uhdm_checkpoint.pth'),
                Path('/kaggle/working/joint_moire_runs/official/uhdm_checkpoint.pth')]
            H.LOCAL_WEIGHT=next((p for p in known if p.is_file()),None)
        model,meta=H.load_official_model()
        if I.S.C.fingerprint(model)!=old_protocol['weight_sha256']:raise RuntimeError('主干权重与冻结融合不匹配')
        assert target_ck['config']['signature']['meta']['weight_sha256']==meta['weight_sha256'],'DCID候选主干权重不一致'
        band=I.S.C.J.ScaleFusion('band',int(band_ck['settings']['preferred'])).cuda().eval()
        band.load_state_dict(band_ck['model'],strict=True);band.requires_grad_(False)
        I.GRID=int(target_ck['config']['grid']);I.MAX_RESIDUAL=float(target_ck['config']['max_residual'])
        repair=I.IndependentRepair('DCID').cuda().eval();repair.load_state_dict(target_ck['model'],strict=True);repair.requires_grad_(False)
        protocol['actual_backbone']=meta;(out/'protocol.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
        infer_all(records,model,band,repair,out)
        print('完成：仅冻结方法测试；不根据测试结果重新调参或重抽样。')
    except Exception as error:
        (out/'error.txt').write_text(repr(error),encoding='utf-8');raise
    finally:I.S.pack(out)
    return out

if __name__=='__main__':
    FROZEN_TEST_RUN=run_frozen_test()
